# 04g Extraction, batch g: ten studies coded from their PDFs and two full-text exclusions

Design log.
- Batch g of notebook 04 (plan v2 B4), coded under the frozen schema (schema/record_schema.yaml, v1) and the coding rules, as batches a to f were. The ten reports: What Survives the Shift from Text to Pixels? (2611.24231), Constitutional Sanitization and Fine-Tuning for log analysis (ICSSA 2026), LLM-as-a-Judge vulnerabilities and a reasoning guardrail (UBMK 2026), Small but Secure (ASIA CCS 2026 poster), Evaluating Mitigation Strategies in RAG (CCS 2026 poster), Are LLM-Agent Prompt-Injection Benchmarks Still Valid? (Research Square), Securing Locally Deployed LLM-RAG Assistants (Applied Sciences), Evaluating Indirect Prompt Injection Defenses in Tool-Using LLM Agents (Computers), The Glass Wall (TechRxiv) and AORT (ETRI Journal).
- Each record's extraction_scope names the tables and rows coded. As before, detectors, sanitisers, filters and system-level designs are left out and the scope says so (amendment 4): Constitutional Sanitization and every condition that contains it, the retrieval filters, MMR and output filter in the RAG poster, the output filter C5 in the local RAG study, the Prompt Injection Detector and Tool Filter in the Computers study. Direct-injection columns are left out too (AORT's TensorTrust, Gandalf, Purple and RuLES), as are injections carried in images (the vision rows of the Glass Wall).
- Coding decisions worth knowing when reading these records (reading aids for notebook 05, not findings):
  - Rendering untrusted text as page images is coded as in Pictionary (axis 1 encodings, axis 2 fixed inference rule).
  - In the log-analysis study the SFT targets were produced under the sanitiser, so the coded SFT defence is sft plus distillation; its benign-log BERTScore is printed with a leading point (.8577) and coded as printed.
  - Several studies print counts rather than rates (log analysis, the RAG poster); those results use unit count with the denominator in the evaluation.
  - AORT's Table 1 prints every decimal point as a colon in the PDF text layer, so its per-model cells cannot be checked against the text; the record codes the five-backbone averages stated in Section 7.1 and the LLaMA 3.1 8B pair stated in the abstract, and a residual note records the loss.
  - The distilled 1B student in the ASIA CCS poster is compared with its 8B teacher on a different base (grade B).
  - Model names printed in the papers are kept as printed, including closed models used as rows (Haiku 5.5 in the text-to-pixels study, Claude Sonnet 4.6 in the Computers study).
- Full-text exclusions found while coding: SafeAgent (preprints202609.1881) was included at title and abstract to check whether its data markers are measured; every configuration tested contains a regex sanitiser with a trained classifier and, beyond Filter Only, a tool allowlist, so it is excluded as system level. The translation defence study (zenodo.18438220) passes the input through a separate translation model before the victim sees it, which is coded as system level as MirrorGuard was in notebook 02d, and most of its corpus is direct injection. Stage 2b checks both anchors in the PDFs and carries the decisions into schema/fulltext_decisions.csv, schema/screening_decisions.csv, reports/screening.csv, reports/defences.csv, reports/fulltext_check.csv and reports/prisma_counts.json, as batches d and f did. The number of included studies falls from 135 to 133.
- Outputs: reports/records/ (ten files), reports/extraction_04g_check.csv, reports/extraction_04g_summary.csv, reports/extraction_batches.csv (rows for batch g added, earlier batches kept), reports/decision_04g.txt.
- CPU; no network calls beyond pip; model calls: 0.

## Stage 0. Session bootstrap

In [ ]:
import os, sys, subprocess, importlib
from pathlib import Path

try:
    from google.colab import drive  # type: ignore
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/SourceMap_Research"
    REPO_DIR = PROJECT_ROOT + "/sourcemap-research"
except ImportError:
    PROJECT_ROOT = os.environ["SOURCEMAP_PROJECT_ROOT"]
    REPO_DIR = os.environ["SOURCEMAP_REPO_ROOT"]
os.environ["SOURCEMAP_PROJECT_ROOT"] = PROJECT_ROOT
os.environ["SOURCEMAP_REPO_ROOT"] = REPO_DIR
os.chdir(REPO_DIR)
if "src" not in sys.path:
    sys.path.insert(0, "src")
for module, package in (("yaml", "pyyaml"), ("pypdf", "pypdf"), ("pypdfium2", "pypdfium2")):
    try:
        importlib.import_module(module)
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", package], check=False)
import pypdf
if int(pypdf.__version__.split(".")[0]) < 4:                 # the column-preserving extraction needs pypdf 4 or later
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pypdf>=4"], check=False)
    print("pypdf upgraded; restart the runtime if the next stage reports an old version")
print(subprocess.run(["git", "pull", "--ff-only"], capture_output=True, text=True).stdout.strip() or "pull: nothing to report")
print("cwd:", os.getcwd(), "| pypdf", pypdf.__version__)

## Stage 1. The 10 records

In [ ]:
Path("reports/records").mkdir(parents=True, exist_ok=True)       # %%writefile does not create folders

`textpixels_arxiv2026` (2611.24231)

In [ ]:
%%writefile reports/records/textpixels_arxiv2026.yaml
report:
  id: textpixels_arxiv2026
  slug: '2611.24231'
  title: What Survives the Shift from Text to Pixels? Conditional Protection against Prompt Injection in Image-Rendered Documents
  first_author: Haoyang Liu
  venue: arXiv preprint
  peer_reviewed: 'no'
  arxiv: '2611.24231'
  version_read: arXiv v1
  first_date: '2026-11-18'
  read: F
  role: re_measures
  code_release: not stated
  extraction_scope: 'Sections 4 and 5.1: Tables 1, 2 and 3; the abstract claims (rendering and wording sweeps of Figure 2, the candidate search of Figure 3, the reading diagnostics of Table 4 and the activation-patching and
    attention-blocking results of Tables 5 and 6 not coded)'
defences:
- id: render_qwen
  name: untrusted source rendered as page images (re-measured)
  origin: other_report
  base_checkpoint: Qwen3-VL-8B-Instruct
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 1
  layout: system policy and question as text; the external source with the injected command rendered as page images (PNG)
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: A promising miti- gation is to render the document as page images
- id: render_gpt
  name: untrusted source rendered as page images (re-measured)
  origin: other_report
  base_checkpoint: GPT-5-nano
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 1
  layout: system policy and question as text; the external source with the injected command rendered as page images (PNG)
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: A promising miti- gation is to render the document as page images
- id: render_haiku
  name: untrusted source rendered as page images (re-measured)
  origin: other_report
  base_checkpoint: Haiku 5.5
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 1
  layout: system policy and question as text; the external source with the injected command rendered as page images (PNG)
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: A promising miti- gation is to render the document as page images
- id: render_gem
  name: untrusted source rendered as page images (re-measured)
  origin: other_report
  base_checkpoint: Gemini 3.1 Flash-Lite
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 1
  layout: system policy and question as text; the external source with the injected command rendered as page images (PNG)
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: A promising miti- gation is to render the document as page images
- id: render_glm
  name: untrusted source rendered as page images (re-measured)
  origin: other_report
  base_checkpoint: GLM-5.3-Flash
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 1
  layout: system policy and question as text; the external source with the injected command rendered as page images (PNG)
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: A promising miti- gation is to render the document as page images
- id: render_ds
  name: untrusted source rendered as page images (re-measured)
  origin: other_report
  base_checkpoint: DeepSeek V4.1 Flash 1017
  axis1:
  - encodings
  axis2:
  - fixed_inference_rule
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 1
  layout: system policy and question as text; the external source with the injected command rendered as page images (PNG)
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: A promising miti- gation is to render the document as page images
evaluations:
- id: ev_qwen_text_append
  defence_id: none:Qwen3-VL-8B-Instruct
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, append a source-specific marker; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_qwen_img_append
  defence_id: render_qwen
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, append a source-specific marker; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_qwen_text_format
  defence_id: none:Qwen3-VL-8B-Instruct
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, format the answer as marker | answer; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_qwen_img_format
  defence_id: render_qwen
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, format the answer as marker | answer; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_qwen_text_replace
  defence_id: none:Qwen3-VL-8B-Instruct
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, replace the answer with a fixed phrase; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_qwen_img_replace
  defence_id: render_qwen
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, replace the answer with a fixed phrase; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_gpt_text_append
  defence_id: none:GPT-5-nano
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, append a source-specific marker; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_gpt_img_append
  defence_id: render_gpt
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, append a source-specific marker; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_gpt_text_format
  defence_id: none:GPT-5-nano
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, format the answer as marker | answer; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_gpt_img_format
  defence_id: render_gpt
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, format the answer as marker | answer; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_gpt_text_replace
  defence_id: none:GPT-5-nano
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, replace the answer with a fixed phrase; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_gpt_img_replace
  defence_id: render_gpt
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, replace the answer with a fixed phrase; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_haiku_text_append
  defence_id: none:Haiku 5.5
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, append a source-specific marker; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_haiku_img_append
  defence_id: render_haiku
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, append a source-specific marker; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_haiku_text_format
  defence_id: none:Haiku 5.5
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, format the answer as marker | answer; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_haiku_img_format
  defence_id: render_haiku
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, format the answer as marker | answer; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_haiku_text_replace
  defence_id: none:Haiku 5.5
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, replace the answer with a fixed phrase; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_haiku_img_replace
  defence_id: render_haiku
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, replace the answer with a fixed phrase; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_gem_text_append
  defence_id: none:Gemini 3.1 Flash-Lite
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, append a source-specific marker; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_gem_img_append
  defence_id: render_gem
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, append a source-specific marker; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_gem_text_format
  defence_id: none:Gemini 3.1 Flash-Lite
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, format the answer as marker | answer; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_gem_img_format
  defence_id: render_gem
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, format the answer as marker | answer; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_gem_text_replace
  defence_id: none:Gemini 3.1 Flash-Lite
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, replace the answer with a fixed phrase; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_gem_img_replace
  defence_id: render_gem
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, replace the answer with a fixed phrase; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_glm_text_append
  defence_id: none:GLM-5.3-Flash
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, append a source-specific marker; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_glm_img_append
  defence_id: render_glm
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, append a source-specific marker; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_glm_text_format
  defence_id: none:GLM-5.3-Flash
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, format the answer as marker | answer; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_glm_img_format
  defence_id: render_glm
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, format the answer as marker | answer; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_glm_text_replace
  defence_id: none:GLM-5.3-Flash
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, replace the answer with a fixed phrase; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_glm_img_replace
  defence_id: render_glm
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, replace the answer with a fixed phrase; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_ds_text_append
  defence_id: none:DeepSeek V4.1 Flash 1017
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, append a source-specific marker; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_ds_img_append
  defence_id: render_ds
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, append a source-specific marker; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_ds_text_format
  defence_id: none:DeepSeek V4.1 Flash 1017
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, format the answer as marker | answer; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_ds_img_format
  defence_id: render_ds
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, format the answer as marker | answer; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_ds_text_replace
  defence_id: none:DeepSeek V4.1 Flash 1017
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, replace the answer with a fixed phrase; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
  baseline_kind: off_the_shelf
- id: ev_ds_img_replace
  defence_id: render_ds
  benchmark: 191 English LongBench sources (Qasper, MultiFieldQA-en, HotpotQA), inline command, wrapper W1, replace the answer with a fixed phrase; two wordings averaged, datasets weighted equally
  attacker_type: static
  attacker_access: black_box
  judge: the visible response carries out the injected operation; truncated responses excluded
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  pages:
    benchmark: 3
- id: ev_qwen_text_orig
  defence_id: none:Qwen3-VL-8B-Instruct
  benchmark: same sources without injection, original LongBench questions (191)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  baseline_kind: off_the_shelf
- id: ev_qwen_img_orig
  defence_id: render_qwen
  benchmark: same sources without injection, original LongBench questions (191)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
- id: ev_qwen_text_integr
  defence_id: none:Qwen3-VL-8B-Instruct
  benchmark: same sources without injection, integration questions needing two evidence spans (191, written by the authors)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  baseline_kind: off_the_shelf
- id: ev_qwen_img_integr
  defence_id: render_qwen
  benchmark: same sources without injection, integration questions needing two evidence spans (191, written by the authors)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
- id: ev_gpt_text_orig
  defence_id: none:GPT-5-nano
  benchmark: same sources without injection, original LongBench questions (191)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  baseline_kind: off_the_shelf
- id: ev_gpt_img_orig
  defence_id: render_gpt
  benchmark: same sources without injection, original LongBench questions (191)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
- id: ev_gpt_text_integr
  defence_id: none:GPT-5-nano
  benchmark: same sources without injection, integration questions needing two evidence spans (191, written by the authors)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  baseline_kind: off_the_shelf
- id: ev_gpt_img_integr
  defence_id: render_gpt
  benchmark: same sources without injection, integration questions needing two evidence spans (191, written by the authors)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
- id: ev_haiku_text_orig
  defence_id: none:Haiku 5.5
  benchmark: same sources without injection, original LongBench questions (191)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  baseline_kind: off_the_shelf
- id: ev_haiku_img_orig
  defence_id: render_haiku
  benchmark: same sources without injection, original LongBench questions (191)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
- id: ev_haiku_text_integr
  defence_id: none:Haiku 5.5
  benchmark: same sources without injection, integration questions needing two evidence spans (191, written by the authors)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  baseline_kind: off_the_shelf
- id: ev_haiku_img_integr
  defence_id: render_haiku
  benchmark: same sources without injection, integration questions needing two evidence spans (191, written by the authors)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
- id: ev_gem_text_orig
  defence_id: none:Gemini 3.1 Flash-Lite
  benchmark: same sources without injection, original LongBench questions (191)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  baseline_kind: off_the_shelf
- id: ev_gem_img_orig
  defence_id: render_gem
  benchmark: same sources without injection, original LongBench questions (191)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
- id: ev_gem_text_integr
  defence_id: none:Gemini 3.1 Flash-Lite
  benchmark: same sources without injection, integration questions needing two evidence spans (191, written by the authors)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  baseline_kind: off_the_shelf
- id: ev_gem_img_integr
  defence_id: render_gem
  benchmark: same sources without injection, integration questions needing two evidence spans (191, written by the authors)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
- id: ev_glm_text_orig
  defence_id: none:GLM-5.3-Flash
  benchmark: same sources without injection, original LongBench questions (191)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  baseline_kind: off_the_shelf
- id: ev_glm_img_orig
  defence_id: render_glm
  benchmark: same sources without injection, original LongBench questions (191)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
- id: ev_glm_text_integr
  defence_id: none:GLM-5.3-Flash
  benchmark: same sources without injection, integration questions needing two evidence spans (191, written by the authors)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  baseline_kind: off_the_shelf
- id: ev_glm_img_integr
  defence_id: render_glm
  benchmark: same sources without injection, integration questions needing two evidence spans (191, written by the authors)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
- id: ev_ds_text_orig
  defence_id: none:DeepSeek V4.1 Flash 1017
  benchmark: same sources without injection, original LongBench questions (191)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  baseline_kind: off_the_shelf
- id: ev_ds_img_orig
  defence_id: render_ds
  benchmark: same sources without injection, original LongBench questions (191)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
- id: ev_ds_text_integr
  defence_id: none:DeepSeek V4.1 Flash 1017
  benchmark: same sources without injection, integration questions needing two evidence spans (191, written by the authors)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
  baseline_kind: off_the_shelf
- id: ev_ds_img_integr
  defence_id: render_ds
  benchmark: same sources without injection, integration questions needing two evidence spans (191, written by the authors)
  attacker_type: none
  attacker_access: not_applicable
  judge: LongBench word-overlap F1, best over references
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '191'
- id: ev_qwen_text_sp_nat_append
  defence_id: none:Qwen3-VL-8B-Instruct
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, append a source-specific marker
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_qwen_img_sp_nat_append
  defence_id: render_qwen
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, append a source-specific marker
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_qwen_text_sp_nat_format
  defence_id: none:Qwen3-VL-8B-Instruct
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, format the answer as marker | answer
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_qwen_img_sp_nat_format
  defence_id: render_qwen
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, format the answer as marker | answer
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_qwen_text_sp_nat_replace
  defence_id: none:Qwen3-VL-8B-Instruct
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, replace the answer with a fixed phrase
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_qwen_img_sp_nat_replace
  defence_id: render_qwen
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, replace the answer with a fixed phrase
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_qwen_text_sp_rnd_append
  defence_id: none:Qwen3-VL-8B-Instruct
  benchmark: 90-source subset, separate-page protocol, random-string target, append a source-specific marker
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_qwen_img_sp_rnd_append
  defence_id: render_qwen
  benchmark: 90-source subset, separate-page protocol, random-string target, append a source-specific marker
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_qwen_text_sp_rnd_format
  defence_id: none:Qwen3-VL-8B-Instruct
  benchmark: 90-source subset, separate-page protocol, random-string target, format the answer as marker | answer
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_qwen_img_sp_rnd_format
  defence_id: render_qwen
  benchmark: 90-source subset, separate-page protocol, random-string target, format the answer as marker | answer
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_qwen_text_sp_rnd_replace
  defence_id: none:Qwen3-VL-8B-Instruct
  benchmark: 90-source subset, separate-page protocol, random-string target, replace the answer with a fixed phrase
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_qwen_img_sp_rnd_replace
  defence_id: render_qwen
  benchmark: 90-source subset, separate-page protocol, random-string target, replace the answer with a fixed phrase
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_gpt_text_sp_nat_append
  defence_id: none:GPT-5-nano
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, append a source-specific marker
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_gpt_img_sp_nat_append
  defence_id: render_gpt
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, append a source-specific marker
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_gpt_text_sp_nat_format
  defence_id: none:GPT-5-nano
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, format the answer as marker | answer
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_gpt_img_sp_nat_format
  defence_id: render_gpt
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, format the answer as marker | answer
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_gpt_text_sp_nat_replace
  defence_id: none:GPT-5-nano
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, replace the answer with a fixed phrase
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_gpt_img_sp_nat_replace
  defence_id: render_gpt
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, replace the answer with a fixed phrase
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_gpt_text_sp_rnd_append
  defence_id: none:GPT-5-nano
  benchmark: 90-source subset, separate-page protocol, random-string target, append a source-specific marker
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_gpt_img_sp_rnd_append
  defence_id: render_gpt
  benchmark: 90-source subset, separate-page protocol, random-string target, append a source-specific marker
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_gpt_text_sp_rnd_format
  defence_id: none:GPT-5-nano
  benchmark: 90-source subset, separate-page protocol, random-string target, format the answer as marker | answer
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_gpt_img_sp_rnd_format
  defence_id: render_gpt
  benchmark: 90-source subset, separate-page protocol, random-string target, format the answer as marker | answer
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_gpt_text_sp_rnd_replace
  defence_id: none:GPT-5-nano
  benchmark: 90-source subset, separate-page protocol, random-string target, replace the answer with a fixed phrase
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_gpt_img_sp_rnd_replace
  defence_id: render_gpt
  benchmark: 90-source subset, separate-page protocol, random-string target, replace the answer with a fixed phrase
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_gem_text_sp_nat_append
  defence_id: none:Gemini 3.1 Flash-Lite
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, append a source-specific marker
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_gem_img_sp_nat_append
  defence_id: render_gem
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, append a source-specific marker
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_gem_text_sp_nat_format
  defence_id: none:Gemini 3.1 Flash-Lite
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, format the answer as marker | answer
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_gem_img_sp_nat_format
  defence_id: render_gem
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, format the answer as marker | answer
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_gem_text_sp_nat_replace
  defence_id: none:Gemini 3.1 Flash-Lite
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, replace the answer with a fixed phrase
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_gem_img_sp_nat_replace
  defence_id: render_gem
  benchmark: 90-source subset, separate-page protocol, natural-phrase target, replace the answer with a fixed phrase
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_gem_text_sp_rnd_append
  defence_id: none:Gemini 3.1 Flash-Lite
  benchmark: 90-source subset, separate-page protocol, random-string target, append a source-specific marker
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_gem_img_sp_rnd_append
  defence_id: render_gem
  benchmark: 90-source subset, separate-page protocol, random-string target, append a source-specific marker
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_gem_text_sp_rnd_format
  defence_id: none:Gemini 3.1 Flash-Lite
  benchmark: 90-source subset, separate-page protocol, random-string target, format the answer as marker | answer
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_gem_img_sp_rnd_format
  defence_id: render_gem
  benchmark: 90-source subset, separate-page protocol, random-string target, format the answer as marker | answer
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
- id: ev_gem_text_sp_rnd_replace
  defence_id: none:Gemini 3.1 Flash-Lite
  benchmark: 90-source subset, separate-page protocol, random-string target, replace the answer with a fixed phrase
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'no'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
  baseline_kind: off_the_shelf
- id: ev_gem_img_sp_rnd_replace
  defence_id: render_gem
  benchmark: 90-source subset, separate-page protocol, random-string target, replace the answer with a fixed phrase
  attacker_type: static
  attacker_access: black_box
  judge: 'strict success: the exact target in the requested structure'
  layout_same_as_defence: 'yes'
  measured_by: textpixels_arxiv2026
  page: 4
  decoding: not reported
  sample_size: '90'
results:
- id: r_qwen_text_append
  evaluation_id: ev_qwen_text_append
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, append
  value: '73.78'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_qwen_img_append
  evaluation_id: ev_qwen_img_append
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, append
  value: '5.01'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_qwen_text_format
  evaluation_id: ev_qwen_text_format
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, format
  value: '71.72'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_qwen_img_format
  evaluation_id: ev_qwen_img_format
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, format
  value: '11.35'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_qwen_text_replace
  evaluation_id: ev_qwen_text_replace
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, replace
  value: '98.73'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_qwen_img_replace
  evaluation_id: ev_qwen_img_replace
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, replace
  value: '77.96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_gpt_text_append
  evaluation_id: ev_gpt_text_append
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, append
  value: '77.95'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_gpt_img_append
  evaluation_id: ev_gpt_img_append
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, append
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_gpt_text_format
  evaluation_id: ev_gpt_text_format
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, format
  value: '72.12'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_gpt_img_format
  evaluation_id: ev_gpt_img_format
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, format
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_gpt_text_replace
  evaluation_id: ev_gpt_text_replace
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, replace
  value: '29.53'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_gpt_img_replace
  evaluation_id: ev_gpt_img_replace
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, replace
  value: '1.12'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_haiku_text_append
  evaluation_id: ev_haiku_text_append
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, append
  value: '2.46'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_haiku_img_append
  evaluation_id: ev_haiku_img_append
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, append
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_haiku_text_format
  evaluation_id: ev_haiku_text_format
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, format
  value: '5.11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_haiku_img_format
  evaluation_id: ev_haiku_img_format
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, format
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_haiku_text_replace
  evaluation_id: ev_haiku_text_replace
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, replace
  value: '0.93'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_haiku_img_replace
  evaluation_id: ev_haiku_img_replace
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, replace
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_gem_text_append
  evaluation_id: ev_gem_text_append
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, append
  value: '93.97'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_gem_img_append
  evaluation_id: ev_gem_img_append
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, append
  value: '47.27'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_gem_text_format
  evaluation_id: ev_gem_text_format
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, format
  value: '97.85'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_gem_img_format
  evaluation_id: ev_gem_img_format
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, format
  value: '59.79'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_gem_text_replace
  evaluation_id: ev_gem_text_replace
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, replace
  value: '30.57'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_gem_img_replace
  evaluation_id: ev_gem_img_replace
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, replace
  value: '0.24'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_glm_text_append
  evaluation_id: ev_glm_text_append
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, append
  value: '65.10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_glm_img_append
  evaluation_id: ev_glm_img_append
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, append
  value: '4.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_glm_text_format
  evaluation_id: ev_glm_text_format
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, format
  value: '61.63'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_glm_img_format
  evaluation_id: ev_glm_img_format
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, format
  value: '1.92'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_glm_text_replace
  evaluation_id: ev_glm_text_replace
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, replace
  value: '0.24'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_glm_img_replace
  evaluation_id: ev_glm_img_replace
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, replace
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_ds_text_append
  evaluation_id: ev_ds_text_append
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, append
  value: '64.87'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_ds_img_append
  evaluation_id: ev_ds_img_append
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, append
  value: '0.61'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_ds_text_format
  evaluation_id: ev_ds_text_format
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, format
  value: '89.26'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_ds_img_format
  evaluation_id: ev_ds_img_format
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, format
  value: '6.69'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_ds_text_replace
  evaluation_id: ev_ds_text_replace
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, replace
  value: '3.60'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_ds_img_replace
  evaluation_id: ev_ds_img_replace
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, replace
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 1
- id: r_qwen_text_orig
  evaluation_id: ev_qwen_text_orig
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, orig
  value: '50.99'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_qwen_img_orig
  evaluation_id: ev_qwen_img_orig
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, orig
  value: '50.33'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_qwen_text_integr
  evaluation_id: ev_qwen_text_integr
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, integr
  value: '59.22'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_qwen_img_integr
  evaluation_id: ev_qwen_img_integr
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, integr
  value: '36.71'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt_text_orig
  evaluation_id: ev_gpt_text_orig
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, orig
  value: '50.11'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt_img_orig
  evaluation_id: ev_gpt_img_orig
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, orig
  value: '49.83'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt_text_integr
  evaluation_id: ev_gpt_text_integr
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, integr
  value: '86.72'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gpt_img_integr
  evaluation_id: ev_gpt_img_integr
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, integr
  value: '77.48'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_haiku_text_orig
  evaluation_id: ev_haiku_text_orig
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, orig
  value: '52.29'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_haiku_img_orig
  evaluation_id: ev_haiku_img_orig
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, orig
  value: '53.13'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_haiku_text_integr
  evaluation_id: ev_haiku_text_integr
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, integr
  value: '81.97'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_haiku_img_integr
  evaluation_id: ev_haiku_img_integr
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, integr
  value: '84.07'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gem_text_orig
  evaluation_id: ev_gem_text_orig
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, orig
  value: '55.18'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gem_img_orig
  evaluation_id: ev_gem_img_orig
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, orig
  value: '55.04'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gem_text_integr
  evaluation_id: ev_gem_text_integr
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, integr
  value: '83.46'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_gem_img_integr
  evaluation_id: ev_gem_img_integr
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, integr
  value: '84.05'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_glm_text_orig
  evaluation_id: ev_glm_text_orig
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, orig
  value: '52.87'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_glm_img_orig
  evaluation_id: ev_glm_img_orig
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, orig
  value: '52.60'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_glm_text_integr
  evaluation_id: ev_glm_text_integr
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, integr
  value: '82.10'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_glm_img_integr
  evaluation_id: ev_glm_img_integr
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, integr
  value: '82.92'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_ds_text_orig
  evaluation_id: ev_ds_text_orig
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, orig
  value: '54.82'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_ds_img_orig
  evaluation_id: ev_ds_img_orig
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, orig
  value: '51.15'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_ds_text_integr
  evaluation_id: ev_ds_text_integr
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, integr
  value: '87.84'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_ds_img_integr
  evaluation_id: ev_ds_img_integr
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean QA F1, integr
  value: '85.13'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 2
- id: r_qwen_text_sp_nat_append
  evaluation_id: ev_qwen_text_sp_nat_append
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, append, natural-phrase target
  value: '73.89'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_qwen_img_sp_nat_append
  evaluation_id: ev_qwen_img_sp_nat_append
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, append, natural-phrase target
  value: '51.11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_qwen_text_sp_nat_format
  evaluation_id: ev_qwen_text_sp_nat_format
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, format, natural-phrase target
  value: '68.89'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_qwen_img_sp_nat_format
  evaluation_id: ev_qwen_img_sp_nat_format
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, format, natural-phrase target
  value: '73.89'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_qwen_text_sp_nat_replace
  evaluation_id: ev_qwen_text_sp_nat_replace
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, replace, natural-phrase target
  value: '79.44'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_qwen_img_sp_nat_replace
  evaluation_id: ev_qwen_img_sp_nat_replace
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, replace, natural-phrase target
  value: '80.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_qwen_text_sp_rnd_append
  evaluation_id: ev_qwen_text_sp_rnd_append
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, append, random-string target
  value: '87.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_qwen_img_sp_rnd_append
  evaluation_id: ev_qwen_img_sp_rnd_append
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, append, random-string target
  value: '59.44'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_qwen_text_sp_rnd_format
  evaluation_id: ev_qwen_text_sp_rnd_format
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, format, random-string target
  value: '94.44'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_qwen_img_sp_rnd_format
  evaluation_id: ev_qwen_img_sp_rnd_format
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, format, random-string target
  value: '77.78'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_qwen_text_sp_rnd_replace
  evaluation_id: ev_qwen_text_sp_rnd_replace
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, replace, random-string target
  value: '95.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_qwen_img_sp_rnd_replace
  evaluation_id: ev_qwen_img_sp_rnd_replace
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, replace, random-string target
  value: '73.33'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gpt_text_sp_nat_append
  evaluation_id: ev_gpt_text_sp_nat_append
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, append, natural-phrase target
  value: '15.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gpt_img_sp_nat_append
  evaluation_id: ev_gpt_img_sp_nat_append
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, append, natural-phrase target
  value: '1.13'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gpt_text_sp_nat_format
  evaluation_id: ev_gpt_text_sp_nat_format
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, format, natural-phrase target
  value: '33.89'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gpt_img_sp_nat_format
  evaluation_id: ev_gpt_img_sp_nat_format
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, format, natural-phrase target
  value: '2.24'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gpt_text_sp_nat_replace
  evaluation_id: ev_gpt_text_sp_nat_replace
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, replace, natural-phrase target
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gpt_img_sp_nat_replace
  evaluation_id: ev_gpt_img_sp_nat_replace
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, replace, natural-phrase target
  value: '0.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gpt_text_sp_rnd_append
  evaluation_id: ev_gpt_text_sp_rnd_append
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, append, random-string target
  value: '21.67'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gpt_img_sp_rnd_append
  evaluation_id: ev_gpt_img_sp_rnd_append
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, append, random-string target
  value: '0.57'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gpt_text_sp_rnd_format
  evaluation_id: ev_gpt_text_sp_rnd_format
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, format, random-string target
  value: '55.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gpt_img_sp_rnd_format
  evaluation_id: ev_gpt_img_sp_rnd_format
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, format, random-string target
  value: '2.26'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gpt_text_sp_rnd_replace
  evaluation_id: ev_gpt_text_sp_rnd_replace
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, replace, random-string target
  value: '1.11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gpt_img_sp_rnd_replace
  evaluation_id: ev_gpt_img_sp_rnd_replace
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, replace, random-string target
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gem_text_sp_nat_append
  evaluation_id: ev_gem_text_sp_nat_append
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, append, natural-phrase target
  value: '80.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gem_img_sp_nat_append
  evaluation_id: ev_gem_img_sp_nat_append
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, append, natural-phrase target
  value: '25.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gem_text_sp_nat_format
  evaluation_id: ev_gem_text_sp_nat_format
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, format, natural-phrase target
  value: '86.11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gem_img_sp_nat_format
  evaluation_id: ev_gem_img_sp_nat_format
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, format, natural-phrase target
  value: '41.67'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gem_text_sp_nat_replace
  evaluation_id: ev_gem_text_sp_nat_replace
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, replace, natural-phrase target
  value: '0.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gem_img_sp_nat_replace
  evaluation_id: ev_gem_img_sp_nat_replace
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, replace, natural-phrase target
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gem_text_sp_rnd_append
  evaluation_id: ev_gem_text_sp_rnd_append
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, append, random-string target
  value: '92.78'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gem_img_sp_rnd_append
  evaluation_id: ev_gem_img_sp_rnd_append
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, append, random-string target
  value: '51.11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gem_text_sp_rnd_format
  evaluation_id: ev_gem_text_sp_rnd_format
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, format, random-string target
  value: '98.89'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gem_img_sp_rnd_format
  evaluation_id: ev_gem_img_sp_rnd_format
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, format, random-string target
  value: '71.11'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gem_text_sp_rnd_replace
  evaluation_id: ev_gem_text_sp_rnd_replace
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, replace, random-string target
  value: '2.22'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
- id: r_gem_img_sp_rnd_replace
  evaluation_id: ev_gem_img_sp_rnd_replace
  quantity: R1
  coverage: direct
  metric_as_printed: strict success, replace, random-string target
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 4
  location: Table 3
comparisons:
- id: c_qwen_append
  left: ev_qwen_img_append
  right: ev_qwen_text_append
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_qwen_format
  left: ev_qwen_img_format
  right: ev_qwen_text_format
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_qwen_replace
  left: ev_qwen_img_replace
  right: ev_qwen_text_replace
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gpt_append
  left: ev_gpt_img_append
  right: ev_gpt_text_append
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gpt_format
  left: ev_gpt_img_format
  right: ev_gpt_text_format
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gpt_replace
  left: ev_gpt_img_replace
  right: ev_gpt_text_replace
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_haiku_append
  left: ev_haiku_img_append
  right: ev_haiku_text_append
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_haiku_format
  left: ev_haiku_img_format
  right: ev_haiku_text_format
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_haiku_replace
  left: ev_haiku_img_replace
  right: ev_haiku_text_replace
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gem_append
  left: ev_gem_img_append
  right: ev_gem_text_append
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gem_format
  left: ev_gem_img_format
  right: ev_gem_text_format
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gem_replace
  left: ev_gem_img_replace
  right: ev_gem_text_replace
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_glm_append
  left: ev_glm_img_append
  right: ev_glm_text_append
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_glm_format
  left: ev_glm_img_format
  right: ev_glm_text_format
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_glm_replace
  left: ev_glm_img_replace
  right: ev_glm_text_replace
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_ds_append
  left: ev_ds_img_append
  right: ev_ds_text_append
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_ds_format
  left: ev_ds_img_format
  right: ev_ds_text_format
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_ds_replace
  left: ev_ds_img_replace
  right: ev_ds_text_replace
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_qwen_orig
  left: ev_qwen_img_orig
  right: ev_qwen_text_orig
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_qwen_integr
  left: ev_qwen_img_integr
  right: ev_qwen_text_integr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gpt_orig
  left: ev_gpt_img_orig
  right: ev_gpt_text_orig
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gpt_integr
  left: ev_gpt_img_integr
  right: ev_gpt_text_integr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_haiku_orig
  left: ev_haiku_img_orig
  right: ev_haiku_text_orig
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_haiku_integr
  left: ev_haiku_img_integr
  right: ev_haiku_text_integr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gem_orig
  left: ev_gem_img_orig
  right: ev_gem_text_orig
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gem_integr
  left: ev_gem_img_integr
  right: ev_gem_text_integr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_glm_orig
  left: ev_glm_img_orig
  right: ev_glm_text_orig
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_glm_integr
  left: ev_glm_img_integr
  right: ev_glm_text_integr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_ds_orig
  left: ev_ds_img_orig
  right: ev_ds_text_orig
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_ds_integr
  left: ev_ds_img_integr
  right: ev_ds_text_integr
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_qwen_sp_nat_append
  left: ev_qwen_img_sp_nat_append
  right: ev_qwen_text_sp_nat_append
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_qwen_sp_nat_format
  left: ev_qwen_img_sp_nat_format
  right: ev_qwen_text_sp_nat_format
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_qwen_sp_nat_replace
  left: ev_qwen_img_sp_nat_replace
  right: ev_qwen_text_sp_nat_replace
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_qwen_sp_rnd_append
  left: ev_qwen_img_sp_rnd_append
  right: ev_qwen_text_sp_rnd_append
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_qwen_sp_rnd_format
  left: ev_qwen_img_sp_rnd_format
  right: ev_qwen_text_sp_rnd_format
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_qwen_sp_rnd_replace
  left: ev_qwen_img_sp_rnd_replace
  right: ev_qwen_text_sp_rnd_replace
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gpt_sp_nat_append
  left: ev_gpt_img_sp_nat_append
  right: ev_gpt_text_sp_nat_append
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gpt_sp_nat_format
  left: ev_gpt_img_sp_nat_format
  right: ev_gpt_text_sp_nat_format
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gpt_sp_nat_replace
  left: ev_gpt_img_sp_nat_replace
  right: ev_gpt_text_sp_nat_replace
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gpt_sp_rnd_append
  left: ev_gpt_img_sp_rnd_append
  right: ev_gpt_text_sp_rnd_append
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gpt_sp_rnd_format
  left: ev_gpt_img_sp_rnd_format
  right: ev_gpt_text_sp_rnd_format
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gpt_sp_rnd_replace
  left: ev_gpt_img_sp_rnd_replace
  right: ev_gpt_text_sp_rnd_replace
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gem_sp_nat_append
  left: ev_gem_img_sp_nat_append
  right: ev_gem_text_sp_nat_append
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gem_sp_nat_format
  left: ev_gem_img_sp_nat_format
  right: ev_gem_text_sp_nat_format
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gem_sp_nat_replace
  left: ev_gem_img_sp_nat_replace
  right: ev_gem_text_sp_nat_replace
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gem_sp_rnd_append
  left: ev_gem_img_sp_rnd_append
  right: ev_gem_text_sp_rnd_append
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gem_sp_rnd_format
  left: ev_gem_img_sp_rnd_format
  right: ev_gem_text_sp_rnd_format
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
- id: c_gem_sp_rnd_replace
  left: ev_gem_img_sp_rnd_replace
  right: ev_gem_text_sp_rnd_replace
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the source presented as page images instead of text
  page: 4
claims:
- id: k1
  text: rendering reduces append and format execution for every model, yet Qwen3-VL-8B still executes image-borne replacement at 77.96%
  location: abstract
  anchor: Rendering reduces append and format execution for every model
  jobs:
  - R1
  supported_by:
  - c_qwen_append
  - c_qwen_format
  - c_qwen_replace
  - c_gpt_append
  - c_gpt_format
  - c_gpt_replace
  - c_haiku_append
  - c_haiku_format
  - c_haiku_replace
  - c_gem_append
  - c_gem_format
  - c_gem_replace
  - c_glm_append
  - c_glm_format
  - c_glm_replace
  - c_ds_append
  - c_ds_format
  - c_ds_replace
  page: 1
  contradicted_by:
  - r_qwen_img_replace
- id: k2
  text: utility ratios on original questions range from 93.31% to 101.61%, but Qwen falls to 61.99% on integration questions
  location: abstract
  anchor: Utility ratios on original questions range from 93.31% to 101.61%
  jobs:
  - R2
  supported_by:
  - c_qwen_orig
  - c_qwen_integr
  - c_gpt_orig
  - c_gpt_integr
  - c_haiku_orig
  - c_haiku_integr
  - c_gem_orig
  - c_gem_integr
  - c_glm_orig
  - c_glm_integr
  - c_ds_orig
  - c_ds_integr
  page: 1
  contradicted_by:
  - r_qwen_img_integr
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 1 and 3
    R2clean: Table 2 (clean QA F1 on the same sources)
    R2attack: not reported
schema_gaps:
- where: report
  note: a re-measurement of rendering untrusted text as images (the Pictionary idea) across six multimodal models; the Haiku 5.5 row is a closed model named as printed; the paper's figures show that larger fonts, higher resolution
    and a search over eight fixed candidates remove much of the protection
- where: evaluations ev_*_sp_*
  note: Table 3 uses a separate-page protocol (commands on their own page with a question-referencing lead-in) on 90 sources, so it is not comparable with Table 1's inline protocol

`constsanitlog_icssa2026` (doi_10_1109_icssa70263_2026_11765792)

In [ ]:
%%writefile reports/records/constsanitlog_icssa2026.yaml
report:
  id: constsanitlog_icssa2026
  slug: doi_10_1109_icssa70263_2026_11765792
  title: Constitutional Sanitization and Fine-Tuning for Robustness to Prompt Injection in LLM-Based Log Analysis
  first_author: Wenkai Ma
  venue: ICSSA 2026 (6th International Conference on Smart Sensors and Applications, IEEE)
  peer_reviewed: 'yes'
  doi: 10.1109/ICSSA70263.2026.11765792
  version_read: IEEE Xplore version
  first_date: '2026-09-01'
  read: F
  role: proposes
  code_release: artifact mentioned; not checked
  extraction_scope: 'Section VI: Table III and Table IV rows for No Defense, Datamarking (Qwen only) and SFT; the introduction claim (Constitutional Sanitization, a sanitiser, and every condition that includes it, Table V
    and Figure 2 not coded)'
defences:
- id: sft_q
  name: LoRA SFT on safe responses from the sanitised pipeline
  origin: this_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: no_ablation
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - 312 training rows (292 attack, 20 normal); attack targets are safe responses produced under Constitutional Sanitization, normal targets are high-utility baseline responses
  - LoRA rank 16, alpha 32, two epochs
  layout: 'single-turn log analysis: one log entry in, one report out (event type, severity, explanation, injection flag)'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: Attack examples preferen- tially use safe final Constitutional Sanitization responses
  pages:
    components: 4
- id: sft_l
  name: LoRA SFT on safe responses from the sanitised pipeline
  origin: this_report
  base_checkpoint: Llama 3.1 8B Instruct
  axis1:
  - no_signal
  axis2:
  - sft
  - distillation
  isolation: no_ablation
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 4
  components:
  - 312 training rows (292 attack, 20 normal); attack targets are safe responses produced under Constitutional Sanitization, normal targets are high-utility baseline responses
  - LoRA rank 16, alpha 32, two epochs
  layout: 'single-turn log analysis: one log entry in, one report out (event type, severity, explanation, injection flag)'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: Attack examples preferen- tially use safe final Constitutional Sanitization responses
  pages:
    components: 4
- id: dm_q
  name: Datamarking (log marked as untrusted data, instruction-like strings framed as data)
  origin: this_report
  base_checkpoint: Qwen2.5-7B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'single-turn log analysis: one log entry in, one report out (event type, severity, explanation, injection flag); log content explicitly boundary-marked'
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: 'Datamarking: log content is explicitly marked as untrusted'
evaluations:
- id: ev_q_none_att
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: frozen external validation set, 600 attack rows (logic splitting, sanitisation spoofing, obfuscated payloads; four goals), rule-based adaptive variants
  attacker_type: static
  attacker_access: black_box
  judge: deterministic goal-specific validator; quoting or analysing the injection does not count
  layout_same_as_defence: 'yes'
  measured_by: constsanitlog_icssa2026
  page: 5
  decoding: not reported
  sample_size: '600'
  denominator: 600 attack rows
  baseline_kind: off_the_shelf
- id: ev_q_none_ben
  defence_id: none:Qwen2.5-7B-Instruct
  benchmark: frozen external validation set, 400 benign logs (160 suspicious-looking hard negatives)
  attacker_type: none
  attacker_access: not_applicable
  judge: BERTScore of the explanation against a gold reference; injection flag and block or refusal from the parsed output
  layout_same_as_defence: 'yes'
  measured_by: constsanitlog_icssa2026
  page: 5
  decoding: not reported
  sample_size: '400'
  denominator: 400 benign rows
  baseline_kind: off_the_shelf
- id: ev_q_dm_att
  defence_id: dm_q
  benchmark: frozen external validation set, 600 attack rows (logic splitting, sanitisation spoofing, obfuscated payloads; four goals), rule-based adaptive variants
  attacker_type: static
  attacker_access: black_box
  judge: deterministic goal-specific validator; quoting or analysing the injection does not count
  layout_same_as_defence: 'yes'
  measured_by: constsanitlog_icssa2026
  page: 5
  decoding: not reported
  sample_size: '600'
  denominator: 600 attack rows
- id: ev_q_dm_ben
  defence_id: dm_q
  benchmark: frozen external validation set, 400 benign logs (160 suspicious-looking hard negatives)
  attacker_type: none
  attacker_access: not_applicable
  judge: BERTScore of the explanation against a gold reference; injection flag and block or refusal from the parsed output
  layout_same_as_defence: 'yes'
  measured_by: constsanitlog_icssa2026
  page: 5
  decoding: not reported
  sample_size: '400'
  denominator: 400 benign rows
- id: ev_q_sft_att
  defence_id: sft_q
  benchmark: frozen external validation set, 600 attack rows (logic splitting, sanitisation spoofing, obfuscated payloads; four goals), rule-based adaptive variants
  attacker_type: static
  attacker_access: black_box
  judge: deterministic goal-specific validator; quoting or analysing the injection does not count
  layout_same_as_defence: 'yes'
  measured_by: constsanitlog_icssa2026
  page: 5
  decoding: not reported
  sample_size: '600'
  denominator: 600 attack rows
- id: ev_q_sft_ben
  defence_id: sft_q
  benchmark: frozen external validation set, 400 benign logs (160 suspicious-looking hard negatives)
  attacker_type: none
  attacker_access: not_applicable
  judge: BERTScore of the explanation against a gold reference; injection flag and block or refusal from the parsed output
  layout_same_as_defence: 'yes'
  measured_by: constsanitlog_icssa2026
  page: 5
  decoding: not reported
  sample_size: '400'
  denominator: 400 benign rows
- id: ev_l_none_att
  defence_id: none:Llama 3.1 8B Instruct
  benchmark: frozen external validation set, 600 attack rows (logic splitting, sanitisation spoofing, obfuscated payloads; four goals), rule-based adaptive variants
  attacker_type: static
  attacker_access: black_box
  judge: deterministic goal-specific validator; quoting or analysing the injection does not count
  layout_same_as_defence: 'yes'
  measured_by: constsanitlog_icssa2026
  page: 5
  decoding: not reported
  sample_size: '600'
  denominator: 600 attack rows
  baseline_kind: off_the_shelf
- id: ev_l_none_ben
  defence_id: none:Llama 3.1 8B Instruct
  benchmark: frozen external validation set, 400 benign logs (160 suspicious-looking hard negatives)
  attacker_type: none
  attacker_access: not_applicable
  judge: BERTScore of the explanation against a gold reference; injection flag and block or refusal from the parsed output
  layout_same_as_defence: 'yes'
  measured_by: constsanitlog_icssa2026
  page: 5
  decoding: not reported
  sample_size: '400'
  denominator: 400 benign rows
  baseline_kind: off_the_shelf
- id: ev_l_sft_att
  defence_id: sft_l
  benchmark: frozen external validation set, 600 attack rows (logic splitting, sanitisation spoofing, obfuscated payloads; four goals), rule-based adaptive variants
  attacker_type: static
  attacker_access: black_box
  judge: deterministic goal-specific validator; quoting or analysing the injection does not count
  layout_same_as_defence: 'yes'
  measured_by: constsanitlog_icssa2026
  page: 5
  decoding: not reported
  sample_size: '600'
  denominator: 600 attack rows
- id: ev_l_sft_ben
  defence_id: sft_l
  benchmark: frozen external validation set, 400 benign logs (160 suspicious-looking hard negatives)
  attacker_type: none
  attacker_access: not_applicable
  judge: BERTScore of the explanation against a gold reference; injection flag and block or refusal from the parsed output
  layout_same_as_defence: 'yes'
  measured_by: constsanitlog_icssa2026
  page: 5
  decoding: not reported
  sample_size: '400'
  denominator: 400 benign rows
results:
- id: r_q_none_asr
  evaluation_id: ev_q_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: attack successes out of 600 (ASR)
  value: '519'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_q_none_bert
  evaluation_id: ev_q_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: explanation BERTScore on benign logs
  value: '.8577'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_q_none_fpr
  evaluation_id: ev_q_none_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign rows flagged as injections (FPR count)
  value: '0'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_q_none_blk
  evaluation_id: ev_q_none_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign rows blocked or refused (count)
  value: '14'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_q_dm_asr
  evaluation_id: ev_q_dm_att
  quantity: R1
  coverage: direct
  metric_as_printed: attack successes out of 600 (ASR)
  value: '383'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_q_dm_bert
  evaluation_id: ev_q_dm_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: explanation BERTScore on benign logs
  value: '.8573'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_q_dm_fpr
  evaluation_id: ev_q_dm_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign rows flagged as injections (FPR count)
  value: '0'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_q_dm_blk
  evaluation_id: ev_q_dm_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign rows blocked or refused (count)
  value: '0'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_q_sft_asr
  evaluation_id: ev_q_sft_att
  quantity: R1
  coverage: direct
  metric_as_printed: attack successes out of 600 (ASR)
  value: '269'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_q_sft_bert
  evaluation_id: ev_q_sft_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: explanation BERTScore on benign logs
  value: '.8611'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_q_sft_fpr
  evaluation_id: ev_q_sft_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign rows flagged as injections (FPR count)
  value: '1'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_q_sft_blk
  evaluation_id: ev_q_sft_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign rows blocked or refused (count)
  value: '17'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_l_none_asr
  evaluation_id: ev_l_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: attack successes out of 600 (ASR)
  value: '329'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_l_none_bert
  evaluation_id: ev_l_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: explanation BERTScore on benign logs
  value: '.8561'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_l_none_fpr
  evaluation_id: ev_l_none_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign rows flagged as injections (FPR count)
  value: '45'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_l_none_blk
  evaluation_id: ev_l_none_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign rows blocked or refused (count)
  value: '5'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_l_sft_asr
  evaluation_id: ev_l_sft_att
  quantity: R1
  coverage: direct
  metric_as_printed: attack successes out of 600 (ASR)
  value: '286'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_l_sft_bert
  evaluation_id: ev_l_sft_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: explanation BERTScore on benign logs
  value: '.8637'
  unit: score
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_l_sft_fpr
  evaluation_id: ev_l_sft_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign rows flagged as injections (FPR count)
  value: '3'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
- id: r_l_sft_blk
  evaluation_id: ev_l_sft_ben
  quantity: other
  coverage: direct
  metric_as_printed: benign rows blocked or refused (count)
  value: '0'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table III
comparisons:
- id: c_q_dm_att
  left: ev_q_dm_att
  right: ev_q_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_q_dm_ben
  left: ev_q_dm_ben
  right: ev_q_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_q_sft_att
  left: ev_q_sft_att
  right: ev_q_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_q_sft_ben
  left: ev_q_sft_ben
  right: ev_q_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_l_sft_att
  left: ev_l_sft_att
  right: ev_l_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_l_sft_ben
  left: ev_l_sft_ben
  right: ev_l_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
claims:
- id: k1
  text: the defences reduce attack success in model-dependent ways, with residual attacks and benign blocking
  location: introduction
  anchor: Results reveal model-dependent ASR reductions
  jobs:
  - R1
  - R2
  supported_by:
  - c_q_sft_att
  - c_l_sft_att
  - c_q_dm_att
  - c_q_sft_ben
  - c_l_sft_ben
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table III (counts out of 600)
    R2clean: Table III (BERTScore on benign logs)
    R2attack: not reported
schema_gaps:
- where: report
  note: the paper's headline defence, Constitutional Sanitization, redacts suspicious spans before the model sees them and is left out as a sanitiser; the SFT targets were themselves produced under sanitisation, so the coded
    SFT defence distils that pipeline
- where: results r_*_asr
  note: Table III prints counts with Wilson intervals; the abstract gives the No Defense rates (86.50% Qwen, 54.83% Llama)

`crosslingjudge_ubmk2026` (doi_10_1109_ubmk71847_2026_11745222)

In [ ]:
%%writefile reports/records/crosslingjudge_ubmk2026.yaml
report:
  id: crosslingjudge_ubmk2026
  slug: doi_10_1109_ubmk71847_2026_11745222
  title: LLM-as-a-Judge Vulnerabilities in Review Summarization and a Reasoning-based Defense against Cross-Lingual Prompt Injection
  first_author: Ali Kerem Güler
  venue: UBMK 2026 (11th International Conference on Computer Science and Engineering, IEEE)
  peer_reviewed: 'yes'
  doi: 10.1109/UBMK71847.2026.11745222
  version_read: IEEE Xplore version
  first_date: '2026-09-01'
  read: F
  role: proposes
  code_release: not stated
  extraction_scope: 'Section IV.B: Table I (ASR in Mode 1 and Mode 2 for six models); the abstract claim (Figure 3 language breakdown, latency, Table II and the judge results of Figure 5, which are figure-only, not coded)'
defences:
- id: guard_gemma2
  name: reasoning guardrail (inspect for language switches and override commands, then summarise)
  origin: this_report
  base_checkpoint: Gemma 2 9B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: no_ablation
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: single-turn summarisation of one review; the guardrail is a system prompt asking for a reasoning block before the summary
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  anchor: In Mode 2 (reasoning guardrail), conversely, a system prompt was prepended
- id: guard_gemma3
  name: reasoning guardrail (inspect for language switches and override commands, then summarise)
  origin: this_report
  base_checkpoint: Gemma 3 4B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: no_ablation
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: single-turn summarisation of one review; the guardrail is a system prompt asking for a reasoning block before the summary
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  anchor: In Mode 2 (reasoning guardrail), conversely, a system prompt was prepended
- id: guard_qwen25
  name: reasoning guardrail (inspect for language switches and override commands, then summarise)
  origin: this_report
  base_checkpoint: Qwen 2.5 7B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: no_ablation
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: single-turn summarisation of one review; the guardrail is a system prompt asking for a reasoning block before the summary
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  anchor: In Mode 2 (reasoning guardrail), conversely, a system prompt was prepended
- id: guard_qwen3
  name: reasoning guardrail (inspect for language switches and override commands, then summarise)
  origin: this_report
  base_checkpoint: Qwen 3 8B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: no_ablation
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: single-turn summarisation of one review; the guardrail is a system prompt asking for a reasoning block before the summary
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  anchor: In Mode 2 (reasoning guardrail), conversely, a system prompt was prepended
- id: guard_llama31
  name: reasoning guardrail (inspect for language switches and override commands, then summarise)
  origin: this_report
  base_checkpoint: Llama 3.1 8B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: no_ablation
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: single-turn summarisation of one review; the guardrail is a system prompt asking for a reasoning block before the summary
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  anchor: In Mode 2 (reasoning guardrail), conversely, a system prompt was prepended
- id: guard_llama32
  name: reasoning guardrail (inspect for language switches and override commands, then summarise)
  origin: this_report
  base_checkpoint: Llama 3.2 3B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: no_ablation
  signal_dependence: not_applicable
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: single-turn summarisation of one review; the guardrail is a system prompt asking for a reasoning block before the summary
  layout_compat:
    user_turn: supported
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  anchor: In Mode 2 (reasoning guardrail), conversely, a system prompt was prepended
evaluations:
- id: ev_gemma2_none
  defence_id: none:Gemma 2 9B
  benchmark: 500 Turkish restaurant reviews (Protel Check and Place), each poisoned with an English, Spanish and Turkish [SYSTEM OVERRIDE] payload demanding a fixed malicious summary
  attacker_type: static
  attacker_access: black_box
  judge: the summary contains a scam-related and a danger-related term in any of the three languages (translations count)
  layout_same_as_defence: 'yes'
  measured_by: crosslingjudge_ubmk2026
  page: 5
  decoding: not reported
  sample_size: 1,500 poisoned inputs per model and mode
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_gemma2_guard
  defence_id: guard_gemma2
  benchmark: 500 Turkish restaurant reviews (Protel Check and Place), each poisoned with an English, Spanish and Turkish [SYSTEM OVERRIDE] payload demanding a fixed malicious summary
  attacker_type: static
  attacker_access: black_box
  judge: the summary contains a scam-related and a danger-related term in any of the three languages (translations count)
  layout_same_as_defence: 'yes'
  measured_by: crosslingjudge_ubmk2026
  page: 5
  decoding: not reported
  sample_size: 1,500 poisoned inputs per model and mode
  pages:
    benchmark: 4
- id: ev_gemma3_none
  defence_id: none:Gemma 3 4B
  benchmark: 500 Turkish restaurant reviews (Protel Check and Place), each poisoned with an English, Spanish and Turkish [SYSTEM OVERRIDE] payload demanding a fixed malicious summary
  attacker_type: static
  attacker_access: black_box
  judge: the summary contains a scam-related and a danger-related term in any of the three languages (translations count)
  layout_same_as_defence: 'yes'
  measured_by: crosslingjudge_ubmk2026
  page: 5
  decoding: not reported
  sample_size: 1,500 poisoned inputs per model and mode
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_gemma3_guard
  defence_id: guard_gemma3
  benchmark: 500 Turkish restaurant reviews (Protel Check and Place), each poisoned with an English, Spanish and Turkish [SYSTEM OVERRIDE] payload demanding a fixed malicious summary
  attacker_type: static
  attacker_access: black_box
  judge: the summary contains a scam-related and a danger-related term in any of the three languages (translations count)
  layout_same_as_defence: 'yes'
  measured_by: crosslingjudge_ubmk2026
  page: 5
  decoding: not reported
  sample_size: 1,500 poisoned inputs per model and mode
  pages:
    benchmark: 4
- id: ev_qwen25_none
  defence_id: none:Qwen 2.5 7B
  benchmark: 500 Turkish restaurant reviews (Protel Check and Place), each poisoned with an English, Spanish and Turkish [SYSTEM OVERRIDE] payload demanding a fixed malicious summary
  attacker_type: static
  attacker_access: black_box
  judge: the summary contains a scam-related and a danger-related term in any of the three languages (translations count)
  layout_same_as_defence: 'yes'
  measured_by: crosslingjudge_ubmk2026
  page: 5
  decoding: not reported
  sample_size: 1,500 poisoned inputs per model and mode
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_qwen25_guard
  defence_id: guard_qwen25
  benchmark: 500 Turkish restaurant reviews (Protel Check and Place), each poisoned with an English, Spanish and Turkish [SYSTEM OVERRIDE] payload demanding a fixed malicious summary
  attacker_type: static
  attacker_access: black_box
  judge: the summary contains a scam-related and a danger-related term in any of the three languages (translations count)
  layout_same_as_defence: 'yes'
  measured_by: crosslingjudge_ubmk2026
  page: 5
  decoding: not reported
  sample_size: 1,500 poisoned inputs per model and mode
  pages:
    benchmark: 4
- id: ev_qwen3_none
  defence_id: none:Qwen 3 8B
  benchmark: 500 Turkish restaurant reviews (Protel Check and Place), each poisoned with an English, Spanish and Turkish [SYSTEM OVERRIDE] payload demanding a fixed malicious summary
  attacker_type: static
  attacker_access: black_box
  judge: the summary contains a scam-related and a danger-related term in any of the three languages (translations count)
  layout_same_as_defence: 'yes'
  measured_by: crosslingjudge_ubmk2026
  page: 5
  decoding: not reported
  sample_size: 1,500 poisoned inputs per model and mode
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_qwen3_guard
  defence_id: guard_qwen3
  benchmark: 500 Turkish restaurant reviews (Protel Check and Place), each poisoned with an English, Spanish and Turkish [SYSTEM OVERRIDE] payload demanding a fixed malicious summary
  attacker_type: static
  attacker_access: black_box
  judge: the summary contains a scam-related and a danger-related term in any of the three languages (translations count)
  layout_same_as_defence: 'yes'
  measured_by: crosslingjudge_ubmk2026
  page: 5
  decoding: not reported
  sample_size: 1,500 poisoned inputs per model and mode
  pages:
    benchmark: 4
- id: ev_llama31_none
  defence_id: none:Llama 3.1 8B
  benchmark: 500 Turkish restaurant reviews (Protel Check and Place), each poisoned with an English, Spanish and Turkish [SYSTEM OVERRIDE] payload demanding a fixed malicious summary
  attacker_type: static
  attacker_access: black_box
  judge: the summary contains a scam-related and a danger-related term in any of the three languages (translations count)
  layout_same_as_defence: 'yes'
  measured_by: crosslingjudge_ubmk2026
  page: 5
  decoding: not reported
  sample_size: 1,500 poisoned inputs per model and mode
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_llama31_guard
  defence_id: guard_llama31
  benchmark: 500 Turkish restaurant reviews (Protel Check and Place), each poisoned with an English, Spanish and Turkish [SYSTEM OVERRIDE] payload demanding a fixed malicious summary
  attacker_type: static
  attacker_access: black_box
  judge: the summary contains a scam-related and a danger-related term in any of the three languages (translations count)
  layout_same_as_defence: 'yes'
  measured_by: crosslingjudge_ubmk2026
  page: 5
  decoding: not reported
  sample_size: 1,500 poisoned inputs per model and mode
  pages:
    benchmark: 4
- id: ev_llama32_none
  defence_id: none:Llama 3.2 3B
  benchmark: 500 Turkish restaurant reviews (Protel Check and Place), each poisoned with an English, Spanish and Turkish [SYSTEM OVERRIDE] payload demanding a fixed malicious summary
  attacker_type: static
  attacker_access: black_box
  judge: the summary contains a scam-related and a danger-related term in any of the three languages (translations count)
  layout_same_as_defence: 'yes'
  measured_by: crosslingjudge_ubmk2026
  page: 5
  decoding: not reported
  sample_size: 1,500 poisoned inputs per model and mode
  pages:
    benchmark: 4
  baseline_kind: off_the_shelf
- id: ev_llama32_guard
  defence_id: guard_llama32
  benchmark: 500 Turkish restaurant reviews (Protel Check and Place), each poisoned with an English, Spanish and Turkish [SYSTEM OVERRIDE] payload demanding a fixed malicious summary
  attacker_type: static
  attacker_access: black_box
  judge: the summary contains a scam-related and a danger-related term in any of the three languages (translations count)
  layout_same_as_defence: 'yes'
  measured_by: crosslingjudge_ubmk2026
  page: 5
  decoding: not reported
  sample_size: 1,500 poisoned inputs per model and mode
  pages:
    benchmark: 4
results:
- id: r_gemma2_none
  evaluation_id: ev_gemma2_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '67.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_gemma2_guard
  evaluation_id: ev_gemma2_guard
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '1.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_gemma3_none
  evaluation_id: ev_gemma3_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '88.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_gemma3_guard
  evaluation_id: ev_gemma3_guard
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '57.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_qwen25_none
  evaluation_id: ev_qwen25_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '60.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_qwen25_guard
  evaluation_id: ev_qwen25_guard
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '20.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_qwen3_none
  evaluation_id: ev_qwen3_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '99.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_qwen3_guard
  evaluation_id: ev_qwen3_guard
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '56.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_llama31_none
  evaluation_id: ev_llama31_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '20.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_llama31_guard
  evaluation_id: ev_llama31_guard
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '13.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_llama32_none
  evaluation_id: ev_llama32_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '5.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_llama32_guard
  evaluation_id: ev_llama32_guard
  quantity: R1
  coverage: direct
  metric_as_printed: ASR
  value: '55.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
comparisons:
- id: c_gemma2
  left: ev_gemma2_guard
  right: ev_gemma2_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_gemma3
  left: ev_gemma3_guard
  right: ev_gemma3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_qwen25
  left: ev_qwen25_guard
  right: ev_qwen25_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_qwen3
  left: ev_qwen3_guard
  right: ev_qwen3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_llama31
  left: ev_llama31_guard
  right: ev_llama31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_llama32
  left: ev_llama32_guard
  right: ev_llama32_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
claims:
- id: k1
  text: the reasoning guardrail reduced attack success on most models by an average of 23.0 points but backfired on the smallest model (5.5% to 55.3%)
  location: abstract
  anchor: The reasoning guardrail sharply reduced successful attacks on most models
  jobs:
  - R1
  supported_by:
  - c_gemma2
  - c_gemma3
  - c_qwen25
  - c_qwen3
  - c_llama31
  - c_llama32
  page: 1
  contradicted_by:
  - r_llama32_guard
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table I
    R2clean: summary quality on clean reviews is not reported (only the detector's false-positive rate)
    R2attack: not reported
schema_gaps:
- where: report
  note: the judge-targeted results (Figure 5) are figure-only; the detector is a keyword rule in three languages and the paper reports a 0.0% false-positive rate on clean reviews

`smallsecure_asiaccs2026` (doi_10_1145_3779208_3804880)

In [ ]:
%%writefile reports/records/smallsecure_asiaccs2026.yaml
report:
  id: smallsecure_asiaccs2026
  slug: doi_10_1145_3779208_3804880
  title: 'POSTER: Small but Secure: Distilling SecAlign Defense on Edge LLMs via On-Policy RL'
  first_author: Debasmita Dey
  venue: ASIA CCS 2026 (poster)
  peer_reviewed: 'yes'
  doi: 10.1145/3779208.3804880
  version_read: ACM published version
  first_date: '2026-06-01'
  read: F
  role: proposes
  code_release: not stated
  extraction_scope: 'Section 4: Table 1 (all columns and rows); the abstract claim'
defences:
- id: onp
  name: on-policy RL distillation of Meta-SecAlign-8B into a 1B student (reverse-KL advantage, PPO clipping)
  origin: this_report
  base_checkpoint: Llama-3.2-1B-Instruct
  axis1:
  - no_signal
  axis2:
  - distillation
  isolation: ablation_present
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 2
  components:
  - student samples responses to injected SecAlign-style inputs; teacher log-probabilities on the same tokens give a per-token negative reverse KL as the advantage
  - PPO-style clipped importance-sampling update, LoRA rank 4, 1,000 training samples, three epochs
  layout: Alpaca-style instruction with data, Llama-3.2 chat template; how instruction and data are separated is not described further
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: We apply the on-policy RL distillation framework for transferring
  pages:
    components: 2
- id: kd
  name: standard knowledge distillation from Meta-SecAlign-8B (forward KL on teacher logits, fixed data)
  origin: this_report
  base_checkpoint: Llama-3.2-1B-Instruct
  axis1:
  - no_signal
  axis2:
  - distillation
  isolation: not_applicable
  signal_dependence: not_applicable
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: Alpaca-style instruction with data, Llama-3.2 chat template; how instruction and data are separated is not described further
  layout_compat: *id001
  anchor: Standard Distillation. We minimize forward KL divergence
- id: msa8
  name: Meta-SecAlign-8B (released checkpoint, the teacher)
  origin: other_report
  base_checkpoint: Llama-3.1-8B-Instruct
  axis1:
  - chat_roles
  axis2:
  - preference_optimisation
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  layout: as released
  anchor: The teacher model is Meta-SecAlign-8B-Instruct
evaluations:
- id: ev_none_ign
  defence_id: none:Llama-3.2-1B-Instruct
  benchmark: AlpacaFarm, 208 samples with a data part, injected instruction Print exactly Hacked!, Ignore attack
  attacker_type: static
  attacker_access: black_box
  judge: the response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: smallsecure_asiaccs2026
  page: 3
  decoding: not reported
  sample_size: '208'
  pages:
    benchmark: 2
  baseline_kind: off_the_shelf
- id: ev_none_comp
  defence_id: none:Llama-3.2-1B-Instruct
  benchmark: AlpacaFarm, 208 samples with a data part, injected instruction Print exactly Hacked!, Completion attack
  attacker_type: static
  attacker_access: black_box
  judge: the response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: smallsecure_asiaccs2026
  page: 3
  decoding: not reported
  sample_size: '208'
  pages:
    benchmark: 2
  baseline_kind: off_the_shelf
- id: ev_none_igc
  defence_id: none:Llama-3.2-1B-Instruct
  benchmark: AlpacaFarm, 208 samples with a data part, injected instruction Print exactly Hacked!, Ignore-Completion attack
  attacker_type: static
  attacker_access: black_box
  judge: the response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: smallsecure_asiaccs2026
  page: 3
  decoding: not reported
  sample_size: '208'
  pages:
    benchmark: 2
  baseline_kind: off_the_shelf
- id: ev_msa8_ign
  defence_id: msa8
  benchmark: AlpacaFarm, 208 samples with a data part, injected instruction Print exactly Hacked!, Ignore attack
  attacker_type: static
  attacker_access: black_box
  judge: the response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: smallsecure_asiaccs2026
  page: 3
  decoding: not reported
  sample_size: '208'
  pages:
    benchmark: 2
- id: ev_msa8_comp
  defence_id: msa8
  benchmark: AlpacaFarm, 208 samples with a data part, injected instruction Print exactly Hacked!, Completion attack
  attacker_type: static
  attacker_access: black_box
  judge: the response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: smallsecure_asiaccs2026
  page: 3
  decoding: not reported
  sample_size: '208'
  pages:
    benchmark: 2
- id: ev_msa8_igc
  defence_id: msa8
  benchmark: AlpacaFarm, 208 samples with a data part, injected instruction Print exactly Hacked!, Ignore-Completion attack
  attacker_type: static
  attacker_access: black_box
  judge: the response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: smallsecure_asiaccs2026
  page: 3
  decoding: not reported
  sample_size: '208'
  pages:
    benchmark: 2
- id: ev_kd_ign
  defence_id: kd
  benchmark: AlpacaFarm, 208 samples with a data part, injected instruction Print exactly Hacked!, Ignore attack
  attacker_type: static
  attacker_access: black_box
  judge: the response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: smallsecure_asiaccs2026
  page: 3
  decoding: not reported
  sample_size: '208'
  pages:
    benchmark: 2
- id: ev_kd_comp
  defence_id: kd
  benchmark: AlpacaFarm, 208 samples with a data part, injected instruction Print exactly Hacked!, Completion attack
  attacker_type: static
  attacker_access: black_box
  judge: the response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: smallsecure_asiaccs2026
  page: 3
  decoding: not reported
  sample_size: '208'
  pages:
    benchmark: 2
- id: ev_kd_igc
  defence_id: kd
  benchmark: AlpacaFarm, 208 samples with a data part, injected instruction Print exactly Hacked!, Ignore-Completion attack
  attacker_type: static
  attacker_access: black_box
  judge: the response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: smallsecure_asiaccs2026
  page: 3
  decoding: not reported
  sample_size: '208'
  pages:
    benchmark: 2
- id: ev_onp_ign
  defence_id: onp
  benchmark: AlpacaFarm, 208 samples with a data part, injected instruction Print exactly Hacked!, Ignore attack
  attacker_type: static
  attacker_access: black_box
  judge: the response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: smallsecure_asiaccs2026
  page: 3
  decoding: not reported
  sample_size: '208'
  pages:
    benchmark: 2
- id: ev_onp_comp
  defence_id: onp
  benchmark: AlpacaFarm, 208 samples with a data part, injected instruction Print exactly Hacked!, Completion attack
  attacker_type: static
  attacker_access: black_box
  judge: the response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: smallsecure_asiaccs2026
  page: 3
  decoding: not reported
  sample_size: '208'
  pages:
    benchmark: 2
- id: ev_onp_igc
  defence_id: onp
  benchmark: AlpacaFarm, 208 samples with a data part, injected instruction Print exactly Hacked!, Ignore-Completion attack
  attacker_type: static
  attacker_access: black_box
  judge: the response begins with Hacked or hacked
  layout_same_as_defence: 'yes'
  measured_by: smallsecure_asiaccs2026
  page: 3
  decoding: not reported
  sample_size: '208'
  pages:
    benchmark: 2
results:
- id: r_none_ign
  evaluation_id: ev_none_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '48.56'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_none_comp
  evaluation_id: ev_none_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '24.04'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_none_igc
  evaluation_id: ev_none_igc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore-Completion
  value: '58.17'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_msa8_ign
  evaluation_id: ev_msa8_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '5.29'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_msa8_comp
  evaluation_id: ev_msa8_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '0.48'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_msa8_igc
  evaluation_id: ev_msa8_igc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore-Completion
  value: '0.00'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_kd_ign
  evaluation_id: ev_kd_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '15.38'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_kd_comp
  evaluation_id: ev_kd_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '17.31'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_kd_igc
  evaluation_id: ev_kd_igc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore-Completion
  value: '29.81'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_onp_ign
  evaluation_id: ev_onp_ign
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore
  value: '0.96'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_onp_comp
  evaluation_id: ev_onp_comp
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Completion
  value: '4.81'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_onp_igc
  evaluation_id: ev_onp_igc
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Ignore-Completion
  value: '3.85'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
comparisons:
- id: c_onp_ign
  left: ev_onp_ign
  right: ev_none_ign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_kd_ign
  left: ev_kd_ign
  right: ev_none_ign
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_onp_kd_ign
  left: ev_onp_ign
  right: ev_kd_ign
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: on-policy sampling with a reverse-KL advantage instead of forward-KL distillation on fixed data
  page: 3
- id: c_onp_msa_ign
  left: ev_onp_ign
  right: ev_msa8_ign
  purpose: W
  c1_base: unmatched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the 1B distilled student against its 8B teacher
  page: 3
- id: c_onp_comp
  left: ev_onp_comp
  right: ev_none_comp
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_kd_comp
  left: ev_kd_comp
  right: ev_none_comp
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_onp_kd_comp
  left: ev_onp_comp
  right: ev_kd_comp
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: on-policy sampling with a reverse-KL advantage instead of forward-KL distillation on fixed data
  page: 3
- id: c_onp_msa_comp
  left: ev_onp_comp
  right: ev_msa8_comp
  purpose: W
  c1_base: unmatched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the 1B distilled student against its 8B teacher
  page: 3
- id: c_onp_igc
  left: ev_onp_igc
  right: ev_none_igc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_kd_igc
  left: ev_kd_igc
  right: ev_none_igc
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_onp_kd_igc
  left: ev_onp_igc
  right: ev_kd_igc
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: on-policy sampling with a reverse-KL advantage instead of forward-KL distillation on fixed data
  page: 3
- id: c_onp_msa_igc
  left: ev_onp_igc
  right: ev_msa8_igc
  purpose: W
  c1_base: unmatched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched_unstated
  varies: the 1B distilled student against its 8B teacher
  page: 3
claims:
- id: k1
  text: on-policy RL distillation clearly beats standard distillation and matches the 8B SecAlign teacher
  location: abstract
  anchor: dramatically outperforms standard knowledge distillation
  jobs:
  - R1
  supported_by:
  - c_onp_kd_ign
  - c_onp_kd_comp
  - c_onp_kd_igc
  - c_onp_msa_ign
  - c_onp_msa_comp
  - c_onp_msa_igc
  - c_onp_ign
  - c_onp_comp
  - c_onp_igc
  page: 1
  contradicted_by:
  - r_onp_comp
  - r_onp_igc
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1
    R2clean: no utility is reported
    general_utility: not reported
schema_gaps:
- where: report
  note: no utility of the distilled student is reported, so the security result cannot be weighed against task performance; the input layout is not described beyond the Llama-3.2 chat template, so axis 1 is coded no_signal
- where: claims k1
  note: against Completion and Ignore-Completion the student (4.81%, 3.85%) is well above the teacher (0.48%, 0.00%)

`ragmitig_ccs2026` (doi_10_1145_3830454_3846434)

In [ ]:
%%writefile reports/records/ragmitig_ccs2026.yaml
report:
  id: ragmitig_ccs2026
  slug: doi_10_1145_3830454_3846434
  title: 'POSTER: Evaluating Mitigation Strategies Against Prompt Injection Attacks in Retrieval-Augmented Language Models'
  first_author: Denis Biewer
  venue: CCS 2026 (poster)
  peer_reviewed: 'yes'
  doi: 10.1145/3830454.3846434
  version_read: ACM published version
  first_date: '2026-11-12'
  read: F
  role: re_measures
  code_release: not stated
  extraction_scope: 'Section 4: Table 1 rows baseline, prompt, prompt_isolation, instruction_hierarchy, grounding and prompt_context_flags, columns instruction-following ASR, final exfiltration leakage, clean pass and refusal;
    the abstract claims (the retrieval-stage defences, retrieval filtering and MMR, and the output filter, which are filters or system-level, and the PoisonedRAG columns, which test knowledge corruption rather than injected
    instructions, not coded)'
defences:
- id: prompt
  name: prompt-stage stack (context isolation, instruction hierarchy, grounding constraints, context flags)
  origin: this_report
  base_checkpoint: Qwen2.5-3B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 2
  layout: 'RAG prompt: top-5 retrieved 256-token chunks inserted into the prompt'
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: Prompt-stage defenses con- trol how the model is instructed to interpret retrieved context
- id: iso
  name: context isolation of retrieved chunks
  origin: this_report
  base_checkpoint: Qwen2.5-3B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 2
  layout: 'RAG prompt: top-5 retrieved 256-token chunks inserted into the prompt'
  layout_compat: *id001
  anchor: Prompt-stage defenses con- trol how the model is instructed to interpret retrieved context
- id: ih
  name: instruction-hierarchy enforcement in the prompt
  origin: this_report
  base_checkpoint: Qwen2.5-3B-Instruct
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 2
  layout: 'RAG prompt: top-5 retrieved 256-token chunks inserted into the prompt'
  layout_compat: *id001
  anchor: Prompt-stage defenses con- trol how the model is instructed to interpret retrieved context
- id: ground
  name: grounding constraints in the prompt
  origin: this_report
  base_checkpoint: Qwen2.5-3B-Instruct
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 2
  layout: 'RAG prompt: top-5 retrieved 256-token chunks inserted into the prompt'
  layout_compat: *id001
  anchor: Prompt-stage defenses con- trol how the model is instructed to interpret retrieved context
- id: flags
  name: prompt context flags on retrieved chunks
  origin: this_report
  base_checkpoint: Qwen2.5-3B-Instruct
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 2
  layout: 'RAG prompt: top-5 retrieved 256-token chunks inserted into the prompt'
  layout_compat: *id001
  anchor: Prompt-stage defenses con- trol how the model is instructed to interpret retrieved context
evaluations:
- id: ev_none_if
  defence_id: none:Qwen2.5-3B-Instruct
  benchmark: 20 HotpotQA-derived close-ended queries, one malicious instruction document per query appended to the corpus (instruction-following attack)
  attacker_type: static
  attacker_access: black_box
  judge: sentinel string contained in the answer
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
  pages:
    benchmark: 2
  baseline_kind: off_the_shelf
- id: ev_none_ex
  defence_id: none:Qwen2.5-3B-Instruct
  benchmark: 20 queries, protected support document with a synthetic access code plus a document instructing the model to reveal it (exfiltration attack)
  attacker_type: static
  attacker_access: black_box
  judge: exact protected code in the user-visible output
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
  pages:
    benchmark: 2
  baseline_kind: off_the_shelf
- id: ev_none_cl
  defence_id: none:Qwen2.5-3B-Instruct
  benchmark: the same 20 queries on the clean corpus
  attacker_type: none
  attacker_access: not_applicable
  judge: benign answer correctness; refusals counted separately
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
  baseline_kind: off_the_shelf
- id: ev_prompt_if
  defence_id: prompt
  benchmark: 20 HotpotQA-derived close-ended queries, one malicious instruction document per query appended to the corpus (instruction-following attack)
  attacker_type: static
  attacker_access: black_box
  judge: sentinel string contained in the answer
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
  pages:
    benchmark: 2
- id: ev_prompt_ex
  defence_id: prompt
  benchmark: 20 queries, protected support document with a synthetic access code plus a document instructing the model to reveal it (exfiltration attack)
  attacker_type: static
  attacker_access: black_box
  judge: exact protected code in the user-visible output
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
  pages:
    benchmark: 2
- id: ev_prompt_cl
  defence_id: prompt
  benchmark: the same 20 queries on the clean corpus
  attacker_type: none
  attacker_access: not_applicable
  judge: benign answer correctness; refusals counted separately
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
- id: ev_iso_if
  defence_id: iso
  benchmark: 20 HotpotQA-derived close-ended queries, one malicious instruction document per query appended to the corpus (instruction-following attack)
  attacker_type: static
  attacker_access: black_box
  judge: sentinel string contained in the answer
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
  pages:
    benchmark: 2
- id: ev_iso_ex
  defence_id: iso
  benchmark: 20 queries, protected support document with a synthetic access code plus a document instructing the model to reveal it (exfiltration attack)
  attacker_type: static
  attacker_access: black_box
  judge: exact protected code in the user-visible output
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
  pages:
    benchmark: 2
- id: ev_iso_cl
  defence_id: iso
  benchmark: the same 20 queries on the clean corpus
  attacker_type: none
  attacker_access: not_applicable
  judge: benign answer correctness; refusals counted separately
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
- id: ev_ih_if
  defence_id: ih
  benchmark: 20 HotpotQA-derived close-ended queries, one malicious instruction document per query appended to the corpus (instruction-following attack)
  attacker_type: static
  attacker_access: black_box
  judge: sentinel string contained in the answer
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
  pages:
    benchmark: 2
- id: ev_ih_ex
  defence_id: ih
  benchmark: 20 queries, protected support document with a synthetic access code plus a document instructing the model to reveal it (exfiltration attack)
  attacker_type: static
  attacker_access: black_box
  judge: exact protected code in the user-visible output
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
  pages:
    benchmark: 2
- id: ev_ih_cl
  defence_id: ih
  benchmark: the same 20 queries on the clean corpus
  attacker_type: none
  attacker_access: not_applicable
  judge: benign answer correctness; refusals counted separately
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
- id: ev_ground_if
  defence_id: ground
  benchmark: 20 HotpotQA-derived close-ended queries, one malicious instruction document per query appended to the corpus (instruction-following attack)
  attacker_type: static
  attacker_access: black_box
  judge: sentinel string contained in the answer
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
  pages:
    benchmark: 2
- id: ev_ground_ex
  defence_id: ground
  benchmark: 20 queries, protected support document with a synthetic access code plus a document instructing the model to reveal it (exfiltration attack)
  attacker_type: static
  attacker_access: black_box
  judge: exact protected code in the user-visible output
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
  pages:
    benchmark: 2
- id: ev_ground_cl
  defence_id: ground
  benchmark: the same 20 queries on the clean corpus
  attacker_type: none
  attacker_access: not_applicable
  judge: benign answer correctness; refusals counted separately
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
- id: ev_flags_if
  defence_id: flags
  benchmark: 20 HotpotQA-derived close-ended queries, one malicious instruction document per query appended to the corpus (instruction-following attack)
  attacker_type: static
  attacker_access: black_box
  judge: sentinel string contained in the answer
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
  pages:
    benchmark: 2
- id: ev_flags_ex
  defence_id: flags
  benchmark: 20 queries, protected support document with a synthetic access code plus a document instructing the model to reveal it (exfiltration attack)
  attacker_type: static
  attacker_access: black_box
  judge: exact protected code in the user-visible output
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
  pages:
    benchmark: 2
- id: ev_flags_cl
  defence_id: flags
  benchmark: the same 20 queries on the clean corpus
  attacker_type: none
  attacker_access: not_applicable
  judge: benign answer correctness; refusals counted separately
  layout_same_as_defence: 'yes'
  measured_by: ragmitig_ccs2026
  page: 3
  decoding: greedy (sampling disabled)
  sample_size: '20'
results:
- id: r_none_if
  evaluation_id: ev_none_if
  quantity: R1
  coverage: direct
  metric_as_printed: instruction-following attack successes out of 20
  value: '17'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_none_ex
  evaluation_id: ev_none_ex
  quantity: R1
  coverage: direct
  metric_as_printed: user-visible leakage out of 20
  value: '19'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_none_cl
  evaluation_id: ev_none_cl
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean queries passed out of 20
  value: '20'
  unit: count
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_none_ref
  evaluation_id: ev_none_cl
  quantity: other
  coverage: direct
  metric_as_printed: clean queries refused out of 20
  value: '0'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_prompt_if
  evaluation_id: ev_prompt_if
  quantity: R1
  coverage: direct
  metric_as_printed: instruction-following attack successes out of 20
  value: '0'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_prompt_ex
  evaluation_id: ev_prompt_ex
  quantity: R1
  coverage: direct
  metric_as_printed: user-visible leakage out of 20
  value: '17'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_prompt_cl
  evaluation_id: ev_prompt_cl
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean queries passed out of 20
  value: '12'
  unit: count
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_prompt_ref
  evaluation_id: ev_prompt_cl
  quantity: other
  coverage: direct
  metric_as_printed: clean queries refused out of 20
  value: '2'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_iso_if
  evaluation_id: ev_iso_if
  quantity: R1
  coverage: direct
  metric_as_printed: instruction-following attack successes out of 20
  value: '10'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_iso_ex
  evaluation_id: ev_iso_ex
  quantity: R1
  coverage: direct
  metric_as_printed: user-visible leakage out of 20
  value: '10'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_iso_cl
  evaluation_id: ev_iso_cl
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean queries passed out of 20
  value: '18'
  unit: count
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_iso_ref
  evaluation_id: ev_iso_cl
  quantity: other
  coverage: direct
  metric_as_printed: clean queries refused out of 20
  value: '1'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_ih_if
  evaluation_id: ev_ih_if
  quantity: R1
  coverage: direct
  metric_as_printed: instruction-following attack successes out of 20
  value: '17'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_ih_ex
  evaluation_id: ev_ih_ex
  quantity: R1
  coverage: direct
  metric_as_printed: user-visible leakage out of 20
  value: '19'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_ih_cl
  evaluation_id: ev_ih_cl
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean queries passed out of 20
  value: '17'
  unit: count
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_ih_ref
  evaluation_id: ev_ih_cl
  quantity: other
  coverage: direct
  metric_as_printed: clean queries refused out of 20
  value: '0'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_ground_if
  evaluation_id: ev_ground_if
  quantity: R1
  coverage: direct
  metric_as_printed: instruction-following attack successes out of 20
  value: '5'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_ground_ex
  evaluation_id: ev_ground_ex
  quantity: R1
  coverage: direct
  metric_as_printed: user-visible leakage out of 20
  value: '17'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_ground_cl
  evaluation_id: ev_ground_cl
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean queries passed out of 20
  value: '17'
  unit: count
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_ground_ref
  evaluation_id: ev_ground_cl
  quantity: other
  coverage: direct
  metric_as_printed: clean queries refused out of 20
  value: '1'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_flags_if
  evaluation_id: ev_flags_if
  quantity: R1
  coverage: direct
  metric_as_printed: instruction-following attack successes out of 20
  value: '15'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_flags_ex
  evaluation_id: ev_flags_ex
  quantity: R1
  coverage: direct
  metric_as_printed: user-visible leakage out of 20
  value: '19'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_flags_cl
  evaluation_id: ev_flags_cl
  quantity: R2clean
  coverage: direct
  metric_as_printed: clean queries passed out of 20
  value: '16'
  unit: count
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
- id: r_flags_ref
  evaluation_id: ev_flags_cl
  quantity: other
  coverage: direct
  metric_as_printed: clean queries refused out of 20
  value: '1'
  unit: count
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 3
  location: Table 1
comparisons:
- id: c_prompt_if
  left: ev_prompt_if
  right: ev_none_if
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_prompt_ex
  left: ev_prompt_ex
  right: ev_none_ex
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_prompt_cl
  left: ev_prompt_cl
  right: ev_none_cl
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_iso_if
  left: ev_iso_if
  right: ev_none_if
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_iso_ex
  left: ev_iso_ex
  right: ev_none_ex
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_iso_cl
  left: ev_iso_cl
  right: ev_none_cl
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_ih_if
  left: ev_ih_if
  right: ev_none_if
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_ih_ex
  left: ev_ih_ex
  right: ev_none_ex
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_ih_cl
  left: ev_ih_cl
  right: ev_none_cl
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_ground_if
  left: ev_ground_if
  right: ev_none_if
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_ground_ex
  left: ev_ground_ex
  right: ev_none_ex
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_ground_cl
  left: ev_ground_cl
  right: ev_none_cl
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_flags_if
  left: ev_flags_if
  right: ev_none_if
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_flags_ex
  left: ev_flags_ex
  right: ev_none_ex
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
- id: c_flags_cl
  left: ev_flags_cl
  right: ev_none_cl
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 3
claims:
- id: k1
  text: retrieval and prompt defences reduced instruction-following success, but no single stage gave broad protection without utility or scope trade-offs
  location: abstract
  anchor: Retrieval and prompt defenses reduced instruction-following success
  jobs:
  - R1
  - R2
  supported_by:
  - c_prompt_if
  - c_prompt_ex
  - c_prompt_cl
  - c_iso_if
  - c_ground_if
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 1 (instruction-following ASR, final exfiltration leakage)
    R2clean: Table 1 (clean pass)
    R2attack: not reported
schema_gaps:
- where: defences flags, iso
  note: the poster names the prompt-stage defences but does not give their wording; what the context flags mark is not described
- where: report
  note: 20 queries per condition; the authors repeated the runs at temperature 0.7 and report 91 to 99% agreement per query, not separate numbers

`benchvalid_rs2026` (doi_10_21203_rs_3_rs_10962146_v1)

In [ ]:
%%writefile reports/records/benchvalid_rs2026.yaml
report:
  id: benchvalid_rs2026
  slug: doi_10_21203_rs_3_rs_10962146_v1
  title: Are LLM-Agent Prompt-Injection Benchmarks Still Valid? A Cross-Model and Cross-Benchmark Study
  first_author: Sushil Khairnar
  venue: Research Square preprint
  peer_reviewed: 'no'
  doi: 10.21203/rs.3.rs-10962146/v1
  version_read: Research Square v1
  first_date: '2026-10-07'
  read: F
  role: re_measures
  code_release: harness stated as open-sourced; not checked
  extraction_scope: 'Section V.D: Table III (defence sweep on the targeted BIPIA-email attack, ten models); the abstract claim on defences (Tables I and II, which test undefended models only, not coded)'
defences:
- id: instr_gemma2
  name: instruction defence (warning that the data is untrusted)
  origin: other_report
  base_checkpoint: Gemma 2 9B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt'
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: untested
    agent_tool_result: untested
  anchor: instruction- defense (with a warning message informing customers
- id: spot_gemma2
  name: spotlighting by delimiting the untrusted data
  origin: other_report
  base_checkpoint: Gemma 2 9B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt; email delimited'
  layout_compat: *id001
  anchor: spotlighting (by delimiting untrusted data)
- id: instr_llama33
  name: instruction defence (warning that the data is untrusted)
  origin: other_report
  base_checkpoint: Llama 3.3 70B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt'
  layout_compat: *id001
  anchor: instruction- defense (with a warning message informing customers
- id: spot_llama33
  name: spotlighting by delimiting the untrusted data
  origin: other_report
  base_checkpoint: Llama 3.3 70B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt; email delimited'
  layout_compat: *id001
  anchor: spotlighting (by delimiting untrusted data)
- id: instr_dsr1
  name: instruction defence (warning that the data is untrusted)
  origin: other_report
  base_checkpoint: DeepSeek R1
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt'
  layout_compat: *id001
  anchor: instruction- defense (with a warning message informing customers
- id: spot_dsr1
  name: spotlighting by delimiting the untrusted data
  origin: other_report
  base_checkpoint: DeepSeek R1
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt; email delimited'
  layout_compat: *id001
  anchor: spotlighting (by delimiting untrusted data)
- id: instr_phi4
  name: instruction defence (warning that the data is untrusted)
  origin: other_report
  base_checkpoint: Phi-4 14B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt'
  layout_compat: *id001
  anchor: instruction- defense (with a warning message informing customers
- id: spot_phi4
  name: spotlighting by delimiting the untrusted data
  origin: other_report
  base_checkpoint: Phi-4 14B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt; email delimited'
  layout_compat: *id001
  anchor: spotlighting (by delimiting untrusted data)
- id: instr_novapro
  name: instruction defence (warning that the data is untrusted)
  origin: other_report
  base_checkpoint: Nova Pro
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt'
  layout_compat: *id001
  anchor: instruction- defense (with a warning message informing customers
- id: spot_novapro
  name: spotlighting by delimiting the untrusted data
  origin: other_report
  base_checkpoint: Nova Pro
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt; email delimited'
  layout_compat: *id001
  anchor: spotlighting (by delimiting untrusted data)
- id: instr_pixtral
  name: instruction defence (warning that the data is untrusted)
  origin: other_report
  base_checkpoint: Mistral Pixtral Large
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt'
  layout_compat: *id001
  anchor: instruction- defense (with a warning message informing customers
- id: spot_pixtral
  name: spotlighting by delimiting the untrusted data
  origin: other_report
  base_checkpoint: Mistral Pixtral Large
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt; email delimited'
  layout_compat: *id001
  anchor: spotlighting (by delimiting untrusted data)
- id: instr_qwen14
  name: instruction defence (warning that the data is untrusted)
  origin: other_report
  base_checkpoint: Qwen2.5 14B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt'
  layout_compat: *id001
  anchor: instruction- defense (with a warning message informing customers
- id: spot_qwen14
  name: spotlighting by delimiting the untrusted data
  origin: other_report
  base_checkpoint: Qwen2.5 14B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt; email delimited'
  layout_compat: *id001
  anchor: spotlighting (by delimiting untrusted data)
- id: instr_mistral7
  name: instruction defence (warning that the data is untrusted)
  origin: other_report
  base_checkpoint: Mistral 7B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt'
  layout_compat: *id001
  anchor: instruction- defense (with a warning message informing customers
- id: spot_mistral7
  name: spotlighting by delimiting the untrusted data
  origin: other_report
  base_checkpoint: Mistral 7B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt; email delimited'
  layout_compat: *id001
  anchor: spotlighting (by delimiting untrusted data)
- id: instr_llama31
  name: instruction defence (warning that the data is untrusted)
  origin: other_report
  base_checkpoint: Llama 3.1 8B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt'
  layout_compat: *id001
  anchor: instruction- defense (with a warning message informing customers
- id: spot_llama31
  name: spotlighting by delimiting the untrusted data
  origin: other_report
  base_checkpoint: Llama 3.1 8B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt; email delimited'
  layout_compat: *id001
  anchor: spotlighting (by delimiting untrusted data)
- id: instr_qwen7
  name: instruction defence (warning that the data is untrusted)
  origin: other_report
  base_checkpoint: Qwen2.5 7B
  axis1:
  - no_signal
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt'
  layout_compat: *id001
  anchor: instruction- defense (with a warning message informing customers
- id: spot_qwen7
  name: spotlighting by delimiting the untrusted data
  origin: other_report
  base_checkpoint: Qwen2.5 7B
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 4
  layout: 'BIPIA email task: the email with the injected sentence request is placed in the prompt; email delimited'
  layout_compat: *id001
  anchor: spotlighting (by delimiting untrusted data)
evaluations:
- id: ev_gemma2_none
  defence_id: none:Gemma 2 9B
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
  baseline_kind: off_the_shelf
- id: ev_gemma2_instr
  defence_id: instr_gemma2
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_gemma2_spot
  defence_id: spot_gemma2
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_llama33_none
  defence_id: none:Llama 3.3 70B
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
  baseline_kind: off_the_shelf
- id: ev_llama33_instr
  defence_id: instr_llama33
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_llama33_spot
  defence_id: spot_llama33
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_dsr1_none
  defence_id: none:DeepSeek R1
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
  baseline_kind: off_the_shelf
- id: ev_dsr1_instr
  defence_id: instr_dsr1
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_dsr1_spot
  defence_id: spot_dsr1
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_phi4_none
  defence_id: none:Phi-4 14B
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
  baseline_kind: off_the_shelf
- id: ev_phi4_instr
  defence_id: instr_phi4
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_phi4_spot
  defence_id: spot_phi4
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_novapro_none
  defence_id: none:Nova Pro
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
  baseline_kind: off_the_shelf
- id: ev_novapro_instr
  defence_id: instr_novapro
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_novapro_spot
  defence_id: spot_novapro
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_pixtral_none
  defence_id: none:Mistral Pixtral Large
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
  baseline_kind: off_the_shelf
- id: ev_pixtral_instr
  defence_id: instr_pixtral
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_pixtral_spot
  defence_id: spot_pixtral
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_qwen14_none
  defence_id: none:Qwen2.5 14B
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
  baseline_kind: off_the_shelf
- id: ev_qwen14_instr
  defence_id: instr_qwen14
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_qwen14_spot
  defence_id: spot_qwen14
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_mistral7_none
  defence_id: none:Mistral 7B
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
  baseline_kind: off_the_shelf
- id: ev_mistral7_instr
  defence_id: instr_mistral7
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_mistral7_spot
  defence_id: spot_mistral7
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_llama31_none
  defence_id: none:Llama 3.1 8B
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
  baseline_kind: off_the_shelf
- id: ev_llama31_instr
  defence_id: instr_llama31
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_llama31_spot
  defence_id: spot_llama31
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_qwen7_none
  defence_id: none:Qwen2.5 7B
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
  baseline_kind: off_the_shelf
- id: ev_qwen7_instr
  defence_id: instr_qwen7
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
- id: ev_qwen7_spot
  defence_id: spot_qwen7
  benchmark: BIPIA email scenarios, one targeted content-insertion injection, 50 cases with two repeats
  attacker_type: static
  attacker_access: black_box
  judge: the target sentence appears in the reply (case-insensitive substring); failed API calls excluded
  layout_same_as_defence: 'yes'
  measured_by: benchvalid_rs2026
  page: 6
  decoding: provider default (no explicit temperature)
  sample_size: 50 cases x 2 repeats
  pages:
    benchmark: 4
    decoding: 4
results:
- id: r_gemma2_none
  evaluation_id: ev_gemma2_none
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_gemma2_instr
  evaluation_id: ev_gemma2_instr
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '70'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_gemma2_spot
  evaluation_id: ev_gemma2_spot
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '15'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_llama33_none
  evaluation_id: ev_llama33_none
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_llama33_instr
  evaluation_id: ev_llama33_instr
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '90'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_llama33_spot
  evaluation_id: ev_llama33_spot
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '97'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_dsr1_none
  evaluation_id: ev_dsr1_none
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '93'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_dsr1_instr
  evaluation_id: ev_dsr1_instr
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '83'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_dsr1_spot
  evaluation_id: ev_dsr1_spot
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '90'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_phi4_none
  evaluation_id: ev_phi4_none
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '75'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_phi4_instr
  evaluation_id: ev_phi4_instr
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '30'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_phi4_spot
  evaluation_id: ev_phi4_spot
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '65'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_novapro_none
  evaluation_id: ev_novapro_none
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '63'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_novapro_instr
  evaluation_id: ev_novapro_instr
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_novapro_spot
  evaluation_id: ev_novapro_spot
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '13'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_pixtral_none
  evaluation_id: ev_pixtral_none
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '53'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_pixtral_instr
  evaluation_id: ev_pixtral_instr
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_pixtral_spot
  evaluation_id: ev_pixtral_spot
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_qwen14_none
  evaluation_id: ev_qwen14_none
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '45'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_qwen14_instr
  evaluation_id: ev_qwen14_instr
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_qwen14_spot
  evaluation_id: ev_qwen14_spot
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_mistral7_none
  evaluation_id: ev_mistral7_none
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '40'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_mistral7_instr
  evaluation_id: ev_mistral7_instr
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_mistral7_spot
  evaluation_id: ev_mistral7_spot
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '10'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_llama31_none
  evaluation_id: ev_llama31_none
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_llama31_instr
  evaluation_id: ev_llama31_instr
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_llama31_spot
  evaluation_id: ev_llama31_spot
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_qwen7_none
  evaluation_id: ev_qwen7_none
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_qwen7_instr
  evaluation_id: ev_qwen7_instr
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
- id: r_qwen7_spot
  evaluation_id: ev_qwen7_spot
  quantity: R1
  coverage: direct
  metric_as_printed: BIPIA targeted-insertion ASR
  value: '0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 6
  location: Table III
comparisons:
- id: c_gemma2_instr
  left: ev_gemma2_instr
  right: ev_gemma2_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_gemma2_spot
  left: ev_gemma2_spot
  right: ev_gemma2_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_llama33_instr
  left: ev_llama33_instr
  right: ev_llama33_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_llama33_spot
  left: ev_llama33_spot
  right: ev_llama33_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_dsr1_instr
  left: ev_dsr1_instr
  right: ev_dsr1_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_dsr1_spot
  left: ev_dsr1_spot
  right: ev_dsr1_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_phi4_instr
  left: ev_phi4_instr
  right: ev_phi4_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_phi4_spot
  left: ev_phi4_spot
  right: ev_phi4_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_novapro_instr
  left: ev_novapro_instr
  right: ev_novapro_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_novapro_spot
  left: ev_novapro_spot
  right: ev_novapro_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_pixtral_instr
  left: ev_pixtral_instr
  right: ev_pixtral_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_pixtral_spot
  left: ev_pixtral_spot
  right: ev_pixtral_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_qwen14_instr
  left: ev_qwen14_instr
  right: ev_qwen14_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_qwen14_spot
  left: ev_qwen14_spot
  right: ev_qwen14_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_mistral7_instr
  left: ev_mistral7_instr
  right: ev_mistral7_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_mistral7_spot
  left: ev_mistral7_spot
  right: ev_mistral7_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_llama31_instr
  left: ev_llama31_instr
  right: ev_llama31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_llama31_spot
  left: ev_llama31_spot
  right: ev_llama31_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_qwen7_instr
  left: ev_qwen7_instr
  right: ev_qwen7_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
- id: c_qwen7_spot
  left: ev_qwen7_spot
  right: ev_qwen7_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 6
claims:
- id: k1
  text: none of the examined prompt-based defences works universally across models
  location: abstract
  anchor: none of the examined prompt-based defenses works universally across models
  jobs:
  - R1
  supported_by:
  - c_gemma2_instr
  - c_gemma2_spot
  - c_llama33_instr
  - c_llama33_spot
  - c_dsr1_instr
  - c_dsr1_spot
  - c_phi4_instr
  - c_phi4_spot
  - c_novapro_instr
  - c_novapro_spot
  - c_pixtral_instr
  - c_pixtral_spot
  - c_qwen14_instr
  - c_qwen14_spot
  - c_mistral7_instr
  - c_mistral7_spot
  - c_llama31_instr
  - c_llama31_spot
  - c_qwen7_instr
  - c_qwen7_spot
  page: 2
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table III
    R2clean: no utility is reported for the defence sweep
    R2attack: not reported
schema_gaps:
- where: evaluations ev_*_none
  note: the no-defence values in Table III differ from Table I for the models in both (DeepSeek R1 93 against 98, Nova Pro 63 against 66, Mistral Pixtral 53 against 38, Llama 3.1 8B 5 against 0), so the two tables come from
    different runs
- where: report
  note: one targeted attack per benchmark; the paper notes that the weakest models' low ASR reflects poor instruction following rather than a defence

`localrag_applsci2026` (doi_10_3390_app162111039)

In [ ]:
%%writefile reports/records/localrag_applsci2026.yaml
report:
  id: localrag_applsci2026
  slug: doi_10_3390_app162111039
  title: 'Securing Locally Deployed LLM-RAG Assistants: A Cross-Model and Cross-Lingual Evaluation of Indirect Prompt Injection Defenses'
  first_author: Sofia Gonzalez-Sancho
  venue: Applied Sciences 16 (2026) 11039
  peer_reviewed: 'yes'
  doi: 10.3390/app162111039
  version_read: published version
  first_date: '2026-11-07'
  read: F
  role: re_measures
  code_release: repository with Docker reproduction (Section 4.2)
  extraction_scope: 'Sections 5.1 and 5.4: configurations C2 (tools, no defence), C3 and C4 in Table 6 (Llama 3.1 8B) and Table 10 (four language conditions), and the English replication values in the text (C1 without tools,
    and C5, a deterministic output filter, not coded; the cross-model C3 and C4 values are in Figure 2 only, and the abstract''s headline claims concern C5)'
defences:
- id: c3
  name: 'C3: retrieved documents structurally delimited and declared untrusted in the system prompt'
  origin: other_report
  base_checkpoint: Llama 3.1 8B (llama3.1:8b, Q4_K_M)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 9
  layout: 'local RAG assistant: top-3 retrieved documents in the prompt, simulated tools requested as TOOL_CALL lines'
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  anchor: 'C3: Explicit instruction–data separation'
- id: c4
  name: 'C4: C3 plus tool-use rules (documents cannot authorise actions) and a self-validation instruction'
  origin: this_report
  base_checkpoint: Llama 3.1 8B (llama3.1:8b, Q4_K_M)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 9
  layout: 'local RAG assistant: top-3 retrieved documents in the prompt, simulated tools requested as TOOL_CALL lines'
  layout_compat: *id001
  anchor: 'C4: Instruction–data separation with tool policy and response validation'
evaluations:
- id: ev_es_none
  defence_id: none:Llama 3.1 8B (llama3.1:8b, Q4_K_M)
  benchmark: Spanish synthetic corpus (40 documents, 32 queries; four attack families), TF-IDF top-3 retrieval
  attacker_type: static
  attacker_access: black_box
  judge: automatic labels (marker and structure rules) with adjudication
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 15
  decoding: temperature 0.0, seed 42
  sample_size: 32 queries (27 exposed to a poisoned document)
  denominator: ASR over the 27 exposed queries; other metrics over all 32
  pages:
    decoding: 12
  baseline_kind: off_the_shelf
- id: ev_es_c3
  defence_id: c3
  benchmark: Spanish synthetic corpus (40 documents, 32 queries; four attack families), TF-IDF top-3 retrieval
  attacker_type: static
  attacker_access: black_box
  judge: automatic labels (marker and structure rules) with adjudication
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 15
  decoding: temperature 0.0, seed 42
  sample_size: 32 queries (27 exposed to a poisoned document)
  denominator: ASR over the 27 exposed queries; other metrics over all 32
  pages:
    decoding: 12
- id: ev_es_c4
  defence_id: c4
  benchmark: Spanish synthetic corpus (40 documents, 32 queries; four attack families), TF-IDF top-3 retrieval
  attacker_type: static
  attacker_access: black_box
  judge: automatic labels (marker and structure rules) with adjudication
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 15
  decoding: temperature 0.0, seed 42
  sample_size: 32 queries (27 exposed to a poisoned document)
  denominator: ASR over the 27 exposed queries; other metrics over all 32
  pages:
    decoding: 12
- id: ev_eses_none
  defence_id: none:Llama 3.1 8B (llama3.1:8b, Q4_K_M)
  benchmark: controlled cross-lingual condition ES-ES (query language, malicious-document language), multilingual embedding retriever
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 20
  decoding: temperature 0.0, seed 42
  sample_size: 28 exposed queries of 32
  baseline_kind: off_the_shelf
- id: ev_eses_c3
  defence_id: c3
  benchmark: controlled cross-lingual condition ES-ES (query language, malicious-document language), multilingual embedding retriever
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 20
  decoding: temperature 0.0, seed 42
  sample_size: 28 exposed queries of 32
- id: ev_eses_c4
  defence_id: c4
  benchmark: controlled cross-lingual condition ES-ES (query language, malicious-document language), multilingual embedding retriever
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 20
  decoding: temperature 0.0, seed 42
  sample_size: 28 exposed queries of 32
- id: ev_esen_none
  defence_id: none:Llama 3.1 8B (llama3.1:8b, Q4_K_M)
  benchmark: controlled cross-lingual condition ES-EN (query language, malicious-document language), multilingual embedding retriever
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 20
  decoding: temperature 0.0, seed 42
  sample_size: 21 exposed queries of 32
  baseline_kind: off_the_shelf
- id: ev_esen_c3
  defence_id: c3
  benchmark: controlled cross-lingual condition ES-EN (query language, malicious-document language), multilingual embedding retriever
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 20
  decoding: temperature 0.0, seed 42
  sample_size: 21 exposed queries of 32
- id: ev_esen_c4
  defence_id: c4
  benchmark: controlled cross-lingual condition ES-EN (query language, malicious-document language), multilingual embedding retriever
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 20
  decoding: temperature 0.0, seed 42
  sample_size: 21 exposed queries of 32
- id: ev_enes_none
  defence_id: none:Llama 3.1 8B (llama3.1:8b, Q4_K_M)
  benchmark: controlled cross-lingual condition EN-ES (query language, malicious-document language), multilingual embedding retriever
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 20
  decoding: temperature 0.0, seed 42
  sample_size: 21 exposed queries of 32
  baseline_kind: off_the_shelf
- id: ev_enes_c3
  defence_id: c3
  benchmark: controlled cross-lingual condition EN-ES (query language, malicious-document language), multilingual embedding retriever
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 20
  decoding: temperature 0.0, seed 42
  sample_size: 21 exposed queries of 32
- id: ev_enes_c4
  defence_id: c4
  benchmark: controlled cross-lingual condition EN-ES (query language, malicious-document language), multilingual embedding retriever
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 20
  decoding: temperature 0.0, seed 42
  sample_size: 21 exposed queries of 32
- id: ev_enen_none
  defence_id: none:Llama 3.1 8B (llama3.1:8b, Q4_K_M)
  benchmark: controlled cross-lingual condition EN-EN (query language, malicious-document language), multilingual embedding retriever
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 20
  decoding: temperature 0.0, seed 42
  sample_size: 27 exposed queries of 32
  baseline_kind: off_the_shelf
- id: ev_enen_c3
  defence_id: c3
  benchmark: controlled cross-lingual condition EN-EN (query language, malicious-document language), multilingual embedding retriever
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 20
  decoding: temperature 0.0, seed 42
  sample_size: 27 exposed queries of 32
- id: ev_enen_c4
  defence_id: c4
  benchmark: controlled cross-lingual condition EN-EN (query language, malicious-document language), multilingual embedding retriever
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 20
  decoding: temperature 0.0, seed 42
  sample_size: 27 exposed queries of 32
- id: ev_eng_none
  defence_id: none:Llama 3.1 8B (llama3.1:8b, Q4_K_M)
  benchmark: independent English replication of the corpus and queries
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 16
  decoding: temperature 0.0, seed 42
  sample_size: 29 exposed queries
  baseline_kind: off_the_shelf
- id: ev_eng_c3
  defence_id: c3
  benchmark: independent English replication of the corpus and queries
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 16
  decoding: temperature 0.0, seed 42
  sample_size: 29 exposed queries
- id: ev_eng_c4
  defence_id: c4
  benchmark: independent English replication of the corpus and queries
  attacker_type: static
  attacker_access: black_box
  judge: extended bilingual automatic evaluator
  layout_same_as_defence: 'yes'
  measured_by: localrag_applsci2026
  page: 16
  decoding: temperature 0.0, seed 42
  sample_size: 29 exposed queries
results:
- id: r_es_none_asr
  evaluation_id: ev_es_none
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR
  value: '74.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_es_none_mis
  evaluation_id: ev_es_none
  quantity: other
  coverage: direct
  metric_as_printed: tool misuse rate (unjustified consequential tool request)
  value: '21.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_es_none_leak
  evaluation_id: ev_es_none
  quantity: other
  coverage: direct
  metric_as_printed: synthetic-data leakage rate
  value: '3.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_es_none_util
  evaluation_id: ev_es_none
  quantity: R2attack
  coverage: direct
  metric_as_printed: answer utility on the poisoned corpus
  value: '53.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_es_c3_asr
  evaluation_id: ev_es_c3
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR
  value: '70.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_es_c3_mis
  evaluation_id: ev_es_c3
  quantity: other
  coverage: direct
  metric_as_printed: tool misuse rate (unjustified consequential tool request)
  value: '34.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_es_c3_leak
  evaluation_id: ev_es_c3
  quantity: other
  coverage: direct
  metric_as_printed: synthetic-data leakage rate
  value: '12.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_es_c3_util
  evaluation_id: ev_es_c3
  quantity: R2attack
  coverage: direct
  metric_as_printed: answer utility on the poisoned corpus
  value: '62.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_es_c4_asr
  evaluation_id: ev_es_c4
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR
  value: '29.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_es_c4_mis
  evaluation_id: ev_es_c4
  quantity: other
  coverage: direct
  metric_as_printed: tool misuse rate (unjustified consequential tool request)
  value: '6.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_es_c4_leak
  evaluation_id: ev_es_c4
  quantity: other
  coverage: direct
  metric_as_printed: synthetic-data leakage rate
  value: '3.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_es_c4_util
  evaluation_id: ev_es_c4
  quantity: R2attack
  coverage: direct
  metric_as_printed: answer utility on the poisoned corpus
  value: '34.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 15
  location: Table 6
- id: r_eses_none
  evaluation_id: ev_eses_none
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, ES-ES
  value: '50.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 10
- id: r_eses_c3
  evaluation_id: ev_eses_c3
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, ES-ES
  value: '57.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 10
- id: r_eses_c4
  evaluation_id: ev_eses_c4
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, ES-ES
  value: '39.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 10
- id: r_esen_none
  evaluation_id: ev_esen_none
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, ES-EN
  value: '47.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 10
- id: r_esen_c3
  evaluation_id: ev_esen_c3
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, ES-EN
  value: '47.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 10
- id: r_esen_c4
  evaluation_id: ev_esen_c4
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, ES-EN
  value: '9.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 10
- id: r_enes_none
  evaluation_id: ev_enes_none
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, EN-ES
  value: '57.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 10
- id: r_enes_c3
  evaluation_id: ev_enes_c3
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, EN-ES
  value: '47.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 10
- id: r_enes_c4
  evaluation_id: ev_enes_c4
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, EN-ES
  value: '42.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 10
- id: r_enen_none
  evaluation_id: ev_enen_none
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, EN-EN
  value: '70.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 10
- id: r_enen_c3
  evaluation_id: ev_enen_c3
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, EN-EN
  value: '63.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 10
- id: r_enen_c4
  evaluation_id: ev_enen_c4
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, EN-EN
  value: '37.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 20
  location: Table 10
- id: r_eng_none
  evaluation_id: ev_eng_none
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, English replication
  value: '55.2'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 16
  location: Section 5.1
- id: r_eng_c3
  evaluation_id: ev_eng_c3
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, English replication
  value: '62.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 16
  location: Section 5.1
- id: r_eng_c4
  evaluation_id: ev_eng_c4
  quantity: R1
  coverage: direct
  metric_as_printed: conditional ASR, English replication
  value: '34.5'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 16
  location: Section 5.1
comparisons:
- id: c_es_c3
  left: ev_es_c3
  right: ev_es_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 15
- id: c_es_c4
  left: ev_es_c4
  right: ev_es_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 15
- id: c_es_c4c3
  left: ev_es_c4
  right: ev_es_c3
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: tool-use rules and self-validation added to the delimiting prompt
  page: 15
- id: c_eses_c3
  left: ev_eses_c3
  right: ev_eses_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_eses_c4
  left: ev_eses_c4
  right: ev_eses_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_eses_c4c3
  left: ev_eses_c4
  right: ev_eses_c3
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: tool-use rules and self-validation added to the delimiting prompt
  page: 20
- id: c_esen_c3
  left: ev_esen_c3
  right: ev_esen_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_esen_c4
  left: ev_esen_c4
  right: ev_esen_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_esen_c4c3
  left: ev_esen_c4
  right: ev_esen_c3
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: tool-use rules and self-validation added to the delimiting prompt
  page: 20
- id: c_enes_c3
  left: ev_enes_c3
  right: ev_enes_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_enes_c4
  left: ev_enes_c4
  right: ev_enes_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_enes_c4c3
  left: ev_enes_c4
  right: ev_enes_c3
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: tool-use rules and self-validation added to the delimiting prompt
  page: 20
- id: c_enen_c3
  left: ev_enen_c3
  right: ev_enen_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_enen_c4
  left: ev_enen_c4
  right: ev_enen_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 20
- id: c_enen_c4c3
  left: ev_enen_c4
  right: ev_enen_c3
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: tool-use rules and self-validation added to the delimiting prompt
  page: 20
- id: c_eng_c3
  left: ev_eng_c3
  right: ev_eng_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
- id: c_eng_c4
  left: ev_eng_c4
  right: ev_eng_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 16
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Tables 6 and 10, Section 5.1
    R2attack: Table 6 (answer utility over all 32 queries on the poisoned corpus)
    R2clean: not reported
schema_gaps:
- where: report
  note: the paper's emphasis is the deterministic output filter C5, left out as a filter; C3 (delimiting) does not lower ASR against C2 in the Llama table and raises tool misuse from 21.9% to 34.4%; the abstract's claims all
    concern C5, so no claim is coded
- where: evaluations ev_es_*
  note: ASR is conditioned on the 27 queries that retrieved a poisoned document, while utility, misuse and leakage use all 32

`ipidefrepl_computers2026` (doi_10_3390_computers15090570)

In [ ]:
%%writefile reports/records/ipidefrepl_computers2026.yaml
report:
  id: ipidefrepl_computers2026
  slug: doi_10_3390_computers15090570
  title: 'Evaluating Indirect Prompt Injection Defenses in Tool-Using LLM Agents: Security, Utility, and Replication'
  first_author: Adil R. Khan
  venue: Computers 15 (2026) 570
  peer_reviewed: 'yes'
  doi: 10.3390/computers15090570
  version_read: published version
  first_date: '2026-08-31'
  read: F
  role: re_measures
  code_release: not stated
  extraction_scope: 'Section 5: Spotlighting and Repeat User Prompt rows with the undefended control in Tables 5 (raw ASR), 7 (benign utility) and 9 (utility under attack) for the three models (the Prompt Injection Detector,
    a detector, and Tool Filter, a tool filter, are left out; the feasibility-conditioned ASR and the per-run tables are not coded)'
defences:
- id: spot_g54
  name: Spotlighting (tool-returned content wrapped in explicit delimiters, AgentDojo implementation)
  origin: other_report
  base_checkpoint: GPT-5.4
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 9
  layout: AgentDojo banking agent; the injection arrives in tool outputs
  layout_compat: &id001
    user_turn: untested
    separate_role: untested
    system_context: untested
    agent_tool_result: supported
  anchor: tool-returned content as external data by wrapping it with explicit delimiters
- id: rep_g54
  name: Repeat User Prompt (the user instruction repeated after tool content)
  origin: other_report
  base_checkpoint: GPT-5.4
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 21
  layout: AgentDojo banking agent; the injection arrives in tool outputs
  layout_compat: *id001
  anchor: Repeat User Prompt reinforces the original user instruction
- id: spot_g54m
  name: Spotlighting (tool-returned content wrapped in explicit delimiters, AgentDojo implementation)
  origin: other_report
  base_checkpoint: GPT-5.4-mini
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 9
  layout: AgentDojo banking agent; the injection arrives in tool outputs
  layout_compat: *id001
  anchor: tool-returned content as external data by wrapping it with explicit delimiters
- id: rep_g54m
  name: Repeat User Prompt (the user instruction repeated after tool content)
  origin: other_report
  base_checkpoint: GPT-5.4-mini
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 21
  layout: AgentDojo banking agent; the injection arrives in tool outputs
  layout_compat: *id001
  anchor: Repeat User Prompt reinforces the original user instruction
- id: spot_cs46
  name: Spotlighting (tool-returned content wrapped in explicit delimiters, AgentDojo implementation)
  origin: other_report
  base_checkpoint: Claude Sonnet 4.6
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 9
  layout: AgentDojo banking agent; the injection arrives in tool outputs
  layout_compat: *id001
  anchor: tool-returned content as external data by wrapping it with explicit delimiters
- id: rep_cs46
  name: Repeat User Prompt (the user instruction repeated after tool content)
  origin: other_report
  base_checkpoint: Claude Sonnet 4.6
  axis1:
  - position
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: baseline_only
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 21
  layout: AgentDojo banking agent; the injection arrives in tool outputs
  layout_compat: *id001
  anchor: Repeat User Prompt reinforces the original user instruction
evaluations:
- id: ev_g54_none_att
  defence_id: none:GPT-5.4
  benchmark: AgentDojo v0.1.35 banking suite, important_instructions attack, two independent replications pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 19
  decoding: temperature 0
  sample_size: 288 attack instances
  pages:
    benchmark: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_g54_none_ben
  defence_id: none:GPT-5.4
  benchmark: AgentDojo v0.1.35 banking suite, no injection, two independent replications pooled
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo user-task check
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 23
  decoding: temperature 0
  sample_size: 32 benign task evaluations
  pages:
    benchmark: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_g54_spot_att
  defence_id: spot_g54
  benchmark: AgentDojo v0.1.35 banking suite, important_instructions attack, two independent replications pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 19
  decoding: temperature 0
  sample_size: 288 attack instances
  pages:
    benchmark: 7
    decoding: 7
- id: ev_g54_spot_ben
  defence_id: spot_g54
  benchmark: AgentDojo v0.1.35 banking suite, no injection, two independent replications pooled
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo user-task check
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 23
  decoding: temperature 0
  sample_size: 32 benign task evaluations
  pages:
    benchmark: 7
    decoding: 7
- id: ev_g54_rep_att
  defence_id: rep_g54
  benchmark: AgentDojo v0.1.35 banking suite, important_instructions attack, two independent replications pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 19
  decoding: temperature 0
  sample_size: 288 attack instances
  pages:
    benchmark: 7
    decoding: 7
- id: ev_g54_rep_ben
  defence_id: rep_g54
  benchmark: AgentDojo v0.1.35 banking suite, no injection, two independent replications pooled
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo user-task check
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 23
  decoding: temperature 0
  sample_size: 32 benign task evaluations
  pages:
    benchmark: 7
    decoding: 7
- id: ev_g54m_none_att
  defence_id: none:GPT-5.4-mini
  benchmark: AgentDojo v0.1.35 banking suite, important_instructions attack, two independent replications pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 19
  decoding: temperature 0
  sample_size: 288 attack instances
  pages:
    benchmark: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_g54m_none_ben
  defence_id: none:GPT-5.4-mini
  benchmark: AgentDojo v0.1.35 banking suite, no injection, two independent replications pooled
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo user-task check
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 23
  decoding: temperature 0
  sample_size: 32 benign task evaluations
  pages:
    benchmark: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_g54m_spot_att
  defence_id: spot_g54m
  benchmark: AgentDojo v0.1.35 banking suite, important_instructions attack, two independent replications pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 19
  decoding: temperature 0
  sample_size: 288 attack instances
  pages:
    benchmark: 7
    decoding: 7
- id: ev_g54m_spot_ben
  defence_id: spot_g54m
  benchmark: AgentDojo v0.1.35 banking suite, no injection, two independent replications pooled
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo user-task check
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 23
  decoding: temperature 0
  sample_size: 32 benign task evaluations
  pages:
    benchmark: 7
    decoding: 7
- id: ev_g54m_rep_att
  defence_id: rep_g54m
  benchmark: AgentDojo v0.1.35 banking suite, important_instructions attack, two independent replications pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 19
  decoding: temperature 0
  sample_size: 288 attack instances
  pages:
    benchmark: 7
    decoding: 7
- id: ev_g54m_rep_ben
  defence_id: rep_g54m
  benchmark: AgentDojo v0.1.35 banking suite, no injection, two independent replications pooled
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo user-task check
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 23
  decoding: temperature 0
  sample_size: 32 benign task evaluations
  pages:
    benchmark: 7
    decoding: 7
- id: ev_cs46_none_att
  defence_id: none:Claude Sonnet 4.6
  benchmark: AgentDojo v0.1.35 banking suite, important_instructions attack, two independent replications pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 19
  decoding: temperature 0
  sample_size: 288 attack instances
  pages:
    benchmark: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_cs46_none_ben
  defence_id: none:Claude Sonnet 4.6
  benchmark: AgentDojo v0.1.35 banking suite, no injection, two independent replications pooled
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo user-task check
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 23
  decoding: temperature 0
  sample_size: 32 benign task evaluations
  pages:
    benchmark: 7
    decoding: 7
  baseline_kind: off_the_shelf
- id: ev_cs46_spot_att
  defence_id: spot_cs46
  benchmark: AgentDojo v0.1.35 banking suite, important_instructions attack, two independent replications pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 19
  decoding: temperature 0
  sample_size: 288 attack instances
  pages:
    benchmark: 7
    decoding: 7
- id: ev_cs46_spot_ben
  defence_id: spot_cs46
  benchmark: AgentDojo v0.1.35 banking suite, no injection, two independent replications pooled
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo user-task check
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 23
  decoding: temperature 0
  sample_size: 32 benign task evaluations
  pages:
    benchmark: 7
    decoding: 7
- id: ev_cs46_rep_att
  defence_id: rep_cs46
  benchmark: AgentDojo v0.1.35 banking suite, important_instructions attack, two independent replications pooled
  attacker_type: static
  attacker_access: black_box
  judge: AgentDojo injection-task and user-task checks
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 19
  decoding: temperature 0
  sample_size: 288 attack instances
  pages:
    benchmark: 7
    decoding: 7
- id: ev_cs46_rep_ben
  defence_id: rep_cs46
  benchmark: AgentDojo v0.1.35 banking suite, no injection, two independent replications pooled
  attacker_type: none
  attacker_access: not_applicable
  judge: AgentDojo user-task check
  layout_same_as_defence: 'yes'
  measured_by: ipidefrepl_computers2026
  page: 23
  decoding: temperature 0
  sample_size: 32 benign task evaluations
  pages:
    benchmark: 7
    decoding: 7
results:
- id: r_g54_none_asr
  evaluation_id: ev_g54_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: raw ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 5
  spread: 0/288 attack successes
- id: r_g54_none_uua
  evaluation_id: ev_g54_none_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '55.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 9
- id: r_g54_none_bu
  evaluation_id: ev_g54_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility
  value: '56.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_g54_spot_asr
  evaluation_id: ev_g54_spot_att
  quantity: R1
  coverage: direct
  metric_as_printed: raw ASR
  value: '0.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 5
  spread: 1/288 attack successes
- id: r_g54_spot_uua
  evaluation_id: ev_g54_spot_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '55.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 9
- id: r_g54_spot_bu
  evaluation_id: ev_g54_spot_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility
  value: '50.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_g54_rep_asr
  evaluation_id: ev_g54_rep_att
  quantity: R1
  coverage: direct
  metric_as_printed: raw ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 5
  spread: 0/288 attack successes
- id: r_g54_rep_uua
  evaluation_id: ev_g54_rep_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '55.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 9
- id: r_g54_rep_bu
  evaluation_id: ev_g54_rep_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility
  value: '50.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_g54m_none_asr
  evaluation_id: ev_g54m_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: raw ASR
  value: '3.8'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 5
  spread: 11/288 attack successes
- id: r_g54m_none_uua
  evaluation_id: ev_g54m_none_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '58.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 9
- id: r_g54m_none_bu
  evaluation_id: ev_g54m_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility
  value: '75.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_g54m_spot_asr
  evaluation_id: ev_g54m_spot_att
  quantity: R1
  coverage: direct
  metric_as_printed: raw ASR
  value: '3.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 5
  spread: 9/288 attack successes
- id: r_g54m_spot_uua
  evaluation_id: ev_g54m_spot_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '58.3'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 9
- id: r_g54m_spot_bu
  evaluation_id: ev_g54m_spot_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility
  value: '59.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_g54m_rep_asr
  evaluation_id: ev_g54m_rep_att
  quantity: R1
  coverage: direct
  metric_as_printed: raw ASR
  value: '1.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 5
  spread: 5/288 attack successes
- id: r_g54m_rep_uua
  evaluation_id: ev_g54m_rep_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '59.0'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 9
- id: r_g54m_rep_bu
  evaluation_id: ev_g54m_rep_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility
  value: '81.2'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_cs46_none_asr
  evaluation_id: ev_cs46_none_att
  quantity: R1
  coverage: direct
  metric_as_printed: raw ASR
  value: '0.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 5
  spread: 1/288 attack successes
- id: r_cs46_none_uua
  evaluation_id: ev_cs46_none_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '71.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 9
- id: r_cs46_none_bu
  evaluation_id: ev_cs46_none_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility
  value: '84.4'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_cs46_spot_asr
  evaluation_id: ev_cs46_spot_att
  quantity: R1
  coverage: direct
  metric_as_printed: raw ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 5
  spread: 0/288 attack successes
- id: r_cs46_spot_uua
  evaluation_id: ev_cs46_spot_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '78.1'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 9
- id: r_cs46_spot_bu
  evaluation_id: ev_cs46_spot_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility
  value: '71.9'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
- id: r_cs46_rep_asr
  evaluation_id: ev_cs46_rep_att
  quantity: R1
  coverage: direct
  metric_as_printed: raw ASR
  value: '0.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 19
  location: Table 5
  spread: 0/288 attack successes
- id: r_cs46_rep_uua
  evaluation_id: ev_cs46_rep_att
  quantity: R2attack
  coverage: direct
  metric_as_printed: utility under attack
  value: '71.5'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 27
  location: Table 9
- id: r_cs46_rep_bu
  evaluation_id: ev_cs46_rep_ben
  quantity: R2clean
  coverage: direct
  metric_as_printed: benign utility
  value: '90.6'
  unit: percent
  direction: higher_better
  qualifier: exact
  value_source: table
  page: 23
  location: Table 7
comparisons:
- id: c_g54_spot_att
  left: ev_g54_spot_att
  right: ev_g54_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_g54_spot_ben
  left: ev_g54_spot_ben
  right: ev_g54_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_g54_rep_att
  left: ev_g54_rep_att
  right: ev_g54_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_g54_rep_ben
  left: ev_g54_rep_ben
  right: ev_g54_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_g54m_spot_att
  left: ev_g54m_spot_att
  right: ev_g54m_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_g54m_spot_ben
  left: ev_g54m_spot_ben
  right: ev_g54m_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_g54m_rep_att
  left: ev_g54m_rep_att
  right: ev_g54m_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_g54m_rep_ben
  left: ev_g54m_rep_ben
  right: ev_g54m_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_cs46_spot_att
  left: ev_cs46_spot_att
  right: ev_cs46_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_cs46_spot_ben
  left: ev_cs46_spot_ben
  right: ev_cs46_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_cs46_rep_att
  left: ev_cs46_rep_att
  right: ev_cs46_none_att
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
- id: c_cs46_rep_ben
  left: ev_cs46_rep_ben
  right: ev_cs46_none_ben
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 19
claims:
- id: k1
  text: defence evaluation should report attack outcomes together with goal feasibility, legitimate-task utility, action availability and run-to-run variation
  location: abstract
  anchor: defense evaluation should report attack outcomes, goal feasibility
  jobs:
  - R1
  - R2
  supported_by:
  - c_g54_spot_att
  - c_g54_spot_ben
  - c_g54_rep_att
  - c_g54_rep_ben
  - c_g54m_spot_att
  - c_g54m_spot_ben
  - c_g54m_rep_att
  - c_g54m_rep_ben
  - c_cs46_spot_att
  - c_cs46_spot_ben
  - c_cs46_rep_att
  - c_cs46_rep_ben
  page: 1
coverage:
  R1: direct
  R2clean: direct
  R2attack: direct
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table 5
    R2clean: Table 7
    R2attack: Table 9
schema_gaps:
- where: evaluations ev_*_att
  note: undefended ASR is near zero for GPT-5.4 (0/288) and Claude Sonnet 4.6 (1/288), and the paper shows that several injection goals were not reachable for those models (Table 6), so their defence comparisons carry little
    information; the model names are as printed
- where: results r_*_uua
  note: Table 9's utility-under-attack values sit on page 27 while the ASR evaluation is anchored on page 19; both pool the same 288 attack instances

`glasswall_techrxiv2026` (doi_10_36227_techrxiv_177219982_22869977_v1)

In [ ]:
%%writefile reports/records/glasswall_techrxiv2026.yaml
report:
  id: glasswall_techrxiv2026
  slug: doi_10_36227_techrxiv_177219982_22869977_v1
  title: 'The Glass Wall: Enforcing Cognitive Segregation in Large Language Models via Zonal LoRA Hardening'
  first_author: Jamil Alshaer
  venue: TechRxiv preprint
  peer_reviewed: 'no'
  doi: 10.36227/techrxiv.177219982.22869977/v1
  version_read: TechRxiv v1
  first_date: '2026-02-27'
  read: F
  role: proposes
  code_release: https://github.com/azimuth-logic-research/Glass-Wall-Protocol (p. 7)
  extraction_scope: 'Section IV: Table I text rows (Phases I to III) and the utility retention stated in Section IV.F; the abstract claim (the vision rows of Table I and Table III, which test injections carried in images,
    and the per-vector Table II, all zero, not coded separately)'
defences:
- id: mw
  name: Glass Wall middleware only (zone sentinels, sentinel sanitisation, hardened system prompt)
  origin: this_report
  base_checkpoint: Qwen-2.5-7B-Instruct (4-bit NF4)
  axis1:
  - delimiters_markers
  axis2:
  - prompting
  isolation: not_applicable
  signal_dependence: not_reported
  trains_instruction_pair: not_applicable
  trains_carry_through: not_applicable
  page: 3
  layout: 'RAG prompt: untrusted data wrapped in <|P_ZONE_START|> and <|P_ZONE_END|> sentinels, closing sentinels inside the data replaced with [BLOCKED], hardened system prompt'
  layout_compat: &id001
    user_turn: supported
    separate_role: untested
    system_context: supported
    agent_tool_result: untested
  anchor: All untrusted data is wrapped in high-entropy sentinels
- id: gw
  name: Glass Wall (middleware plus a LoRA adapter on the query and value projections)
  origin: this_report
  base_checkpoint: Qwen-2.5-7B-Instruct (4-bit NF4)
  axis1:
  - delimiters_markers
  axis2:
  - sft
  isolation: ablation_present
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 3
  components:
  - zone sentinels with sentinel sanitisation and a hardened system prompt
  - LoRA on Q and V projections trained for 15 steps on adversarial samples
  layout: 'RAG prompt: untrusted data wrapped in <|P_ZONE_START|> and <|P_ZONE_END|> sentinels, closing sentinels inside the data replaced with [BLOCKED], hardened system prompt'
  layout_compat: *id001
  anchor: By exposing the model to adversarial samples during a 15-step
  pages:
    components: 3
evaluations:
- id: ev_p1_none
  defence_id: none:Qwen-2.5-7B-Instruct (4-bit NF4)
  benchmark: 'author-built RAG injection suite, Phase I baseline vulnerability: five basic attack vectors'
  attacker_type: static
  attacker_access: black_box
  judge: leak detected in the response (detector not described)
  layout_same_as_defence: 'yes'
  measured_by: glasswall_techrxiv2026
  page: 5
  decoding: not reported
  sample_size: 10,000 iterations
  baseline_kind: off_the_shelf
- id: ev_p1_mw
  defence_id: mw
  benchmark: 'author-built RAG injection suite, Phase I baseline vulnerability: five basic attack vectors'
  attacker_type: static
  attacker_access: black_box
  judge: leak detected in the response (detector not described)
  layout_same_as_defence: 'yes'
  measured_by: glasswall_techrxiv2026
  page: 5
  decoding: not reported
  sample_size: 10,000 iterations
- id: ev_p2_none
  defence_id: none:Qwen-2.5-7B-Instruct (4-bit NF4)
  benchmark: author-built RAG injection suite, Phase II initial stress test
  attacker_type: static
  attacker_access: black_box
  judge: leak detected in the response (detector not described)
  layout_same_as_defence: 'yes'
  measured_by: glasswall_techrxiv2026
  page: 5
  decoding: not reported
  sample_size: 1,100 iterations
  baseline_kind: off_the_shelf
- id: ev_p2_gw
  defence_id: gw
  benchmark: author-built RAG injection suite, Phase II initial stress test
  attacker_type: static
  attacker_access: black_box
  judge: leak detected in the response (detector not described)
  layout_same_as_defence: 'yes'
  measured_by: glasswall_techrxiv2026
  page: 5
  decoding: not reported
  sample_size: 1,100 iterations
- id: ev_p3_none
  defence_id: none:Qwen-2.5-7B-Instruct (4-bit NF4)
  benchmark: 'author-built RAG injection suite, Phase III stratified certification: ten attack vectors including recursive logic and Arabic injections'
  attacker_type: static
  attacker_access: black_box
  judge: leak detected in the response (detector not described)
  layout_same_as_defence: 'yes'
  measured_by: glasswall_techrxiv2026
  page: 5
  decoding: not reported
  sample_size: 10,000 iterations
  baseline_kind: off_the_shelf
- id: ev_p3_gw
  defence_id: gw
  benchmark: 'author-built RAG injection suite, Phase III stratified certification: ten attack vectors including recursive logic and Arabic injections'
  attacker_type: static
  attacker_access: black_box
  judge: leak detected in the response (detector not described)
  layout_same_as_defence: 'yes'
  measured_by: glasswall_techrxiv2026
  page: 5
  decoding: not reported
  sample_size: 10,000 iterations
- id: ev_util_gw
  defence_id: gw
  benchmark: standard summarisation benchmarks (not named)
  attacker_type: none
  attacker_access: not_applicable
  judge: not stated
  layout_same_as_defence: 'yes'
  measured_by: glasswall_techrxiv2026
  page: 5
  decoding: not reported
results:
- id: r_p1_none
  evaluation_id: ev_p1_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Phase I baseline vulnerability
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_p1_mw
  evaluation_id: ev_p1_mw
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Phase I baseline vulnerability
  value: '21.17'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_p2_none
  evaluation_id: ev_p2_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Phase II initial stress test
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_p2_gw
  evaluation_id: ev_p2_gw
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Phase II initial stress test
  value: '0.000'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_p3_none
  evaluation_id: ev_p3_none
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Phase III stratified certification
  value: '100'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_p3_gw
  evaluation_id: ev_p3_gw
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, Phase III stratified certification
  value: '0.000'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: table
  page: 5
  location: Table I
- id: r_util_gw
  evaluation_id: ev_util_gw
  quantity: general_utility
  coverage: proxy
  metric_as_printed: summarisation performance retained
  value: '98.2'
  unit: percent_of_reference
  direction: higher_better
  qualifier: exact
  value_source: text
  page: 5
  location: Section IV.F
  reference: the unhardened model
comparisons:
- id: c_p1_mw
  left: ev_p1_mw
  right: ev_p1_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_p2_gw
  left: ev_p2_gw
  right: ev_p2_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_p3_gw
  left: ev_p3_gw
  right: ev_p3_none
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the defence (defended against undefended arm)
  page: 5
- id: c_lora
  left: ev_p3_gw
  right: ev_p1_mw
  purpose: M
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: unmatched
  c5_judge: matched_unstated
  c6_decoding: matched_unstated
  varies: the LoRA adapter added to the middleware (Phase III against Phase I, different attack sets)
  page: 5
claims:
- id: k1
  text: the Glass Wall moves text injection from a 100% baseline failure rate to 0.00% ASR across all tested vectors
  location: abstract
  anchor: a transition from a 100% baseline failure rate to a deterministic 0.00% ASR
  jobs:
  - R1
  supported_by:
  - c_p3_gw
  - c_p2_gw
  - c_lora
  page: 1
coverage:
  R1: direct
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Table I
    general_utility: Section IV.F (98.2% retained, benchmark not named)
    R2clean: not reported
    R2attack: not reported
schema_gaps:
- where: report
  note: the attack suite, detector and LoRA training data are author-built and only briefly described; the unprotected arm is reported as 100% in every phase, the middleware-only and full arms were tested on different attack
    sets, and the 0.000% claims are not checked against any public benchmark

`aort_etrij2026` (doi_10_4218_etrij_2025_0449)

In [ ]:
%%writefile reports/records/aort_etrij2026.yaml
report:
  id: aort_etrij2026
  slug: doi_10_4218_etrij_2025_0449
  title: Adaptive orthogonal role-aware transformer for robust instruction alignment
  first_author: Yeo Chan Yoon
  venue: ETRI Journal 48(5) (2026) 924-935
  peer_reviewed: 'yes'
  doi: 10.4218/etrij.2025-0449
  version_read: published version (Wiley)
  first_date: '2026-06-24'
  read: F
  role: proposes
  code_release: not stated
  extraction_scope: 'Section 7.1: the Vanilla and AORT averages over five backbones stated in the text for the four indirect benchmarks, and the LLaMA 3.1 8B BIPIA-text values stated in the abstract; the abstract claim (the
    per-model cells of Table 1, including the ISE and ASIDE rows, are not coded because the PDF text layer prints every decimal point in that table as a colon; the direct-injection columns are out of scope; Tables 2 to 4 are
    mechanism analyses)'
defences:
- id: aort_avg
  name: AORT (adaptive orthogonal role-aware transformer)
  origin: this_report
  base_checkpoint: five backbones averaged (LLaMA 2 7B, LLaMA 2 13B, LLaMA 3.1 8B, Qwen2.5 7B, Mistral 7B)
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - sft
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components: &id001
  - 'role mask from the chat template: system and developer text is instruction; user turns, retrieved documents and tool outputs are data'
  - at every layer, data tokens are re-encoded with a learned orthonormal transform and instruction keys get a learned additive attention bias
  - backbone frozen; AORT parameters trained with next-token loss plus cross-role margin, decorrelation and smoothness terms on the same instruction data as the baselines
  layout: chat template with system/developer instruction segments and user or retrieved data segments
  layout_compat: &id002
    user_turn: untested
    separate_role: supported
    system_context: supported
    agent_tool_result: untested
  anchor: This structure was used to supply a binary role mask
  pages:
    components: 6
- id: aort_l31
  name: AORT (adaptive orthogonal role-aware transformer)
  origin: this_report
  base_checkpoint: LLaMA 3.1 8B
  axis1:
  - chat_roles
  - internal_intervention
  axis2:
  - sft
  isolation: no_ablation
  signal_dependence: not_reported
  trains_instruction_pair: 'no'
  trains_carry_through: 'no'
  page: 6
  components: *id001
  layout: chat template with system/developer instruction segments and user or retrieved data segments
  layout_compat: *id002
  anchor: This structure was used to supply a binary role mask
  pages:
    components: 6
evaluations:
- id: ev_avg_none_bipiatext
  defence_id: none:five backbones averaged (LLaMA 2 7B, LLaMA 2 13B, LLaMA 3.1 8B, Qwen2.5 7B, Mistral 7B)
  benchmark: BIPIA-text (indirect injection), averaged over the five backbones
  attacker_type: static
  attacker_access: black_box
  judge: benchmark success criteria (Supplementary Section S1)
  layout_same_as_defence: 'yes'
  measured_by: aort_etrij2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_avg_aort_bipiatext
  defence_id: aort_avg
  benchmark: BIPIA-text (indirect injection), averaged over the five backbones
  attacker_type: static
  attacker_access: black_box
  judge: benchmark success criteria (Supplementary Section S1)
  layout_same_as_defence: 'yes'
  measured_by: aort_etrij2026
  page: 7
  decoding: not reported
- id: ev_avg_none_bipiacode
  defence_id: none:five backbones averaged (LLaMA 2 7B, LLaMA 2 13B, LLaMA 3.1 8B, Qwen2.5 7B, Mistral 7B)
  benchmark: BIPIA-code (indirect injection), averaged over the five backbones
  attacker_type: static
  attacker_access: black_box
  judge: benchmark success criteria (Supplementary Section S1)
  layout_same_as_defence: 'yes'
  measured_by: aort_etrij2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_avg_aort_bipiacode
  defence_id: aort_avg
  benchmark: BIPIA-code (indirect injection), averaged over the five backbones
  attacker_type: static
  attacker_access: black_box
  judge: benchmark success criteria (Supplementary Section S1)
  layout_same_as_defence: 'yes'
  measured_by: aort_etrij2026
  page: 7
  decoding: not reported
- id: ev_avg_none_struqid
  defence_id: none:five backbones averaged (LLaMA 2 7B, LLaMA 2 13B, LLaMA 3.1 8B, Qwen2.5 7B, Mistral 7B)
  benchmark: StruQ-ID (indirect injection), averaged over the five backbones
  attacker_type: static
  attacker_access: black_box
  judge: benchmark success criteria (Supplementary Section S1)
  layout_same_as_defence: 'yes'
  measured_by: aort_etrij2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_avg_aort_struqid
  defence_id: aort_avg
  benchmark: StruQ-ID (indirect injection), averaged over the five backbones
  attacker_type: static
  attacker_access: black_box
  judge: benchmark success criteria (Supplementary Section S1)
  layout_same_as_defence: 'yes'
  measured_by: aort_etrij2026
  page: 7
  decoding: not reported
- id: ev_avg_none_struqood
  defence_id: none:five backbones averaged (LLaMA 2 7B, LLaMA 2 13B, LLaMA 3.1 8B, Qwen2.5 7B, Mistral 7B)
  benchmark: StruQ-OOD (indirect injection), averaged over the five backbones
  attacker_type: static
  attacker_access: black_box
  judge: benchmark success criteria (Supplementary Section S1)
  layout_same_as_defence: 'yes'
  measured_by: aort_etrij2026
  page: 7
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_avg_aort_struqood
  defence_id: aort_avg
  benchmark: StruQ-OOD (indirect injection), averaged over the five backbones
  attacker_type: static
  attacker_access: black_box
  judge: benchmark success criteria (Supplementary Section S1)
  layout_same_as_defence: 'yes'
  measured_by: aort_etrij2026
  page: 7
  decoding: not reported
- id: ev_l31_none_bipiatext
  defence_id: none:LLaMA 3.1 8B
  benchmark: BIPIA-text (indirect injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark success criteria (Supplementary Section S1)
  layout_same_as_defence: 'yes'
  measured_by: aort_etrij2026
  page: 1
  decoding: not reported
  baseline_kind: same_data_twin
- id: ev_l31_aort_bipiatext
  defence_id: aort_l31
  benchmark: BIPIA-text (indirect injection)
  attacker_type: static
  attacker_access: black_box
  judge: benchmark success criteria (Supplementary Section S1)
  layout_same_as_defence: 'yes'
  measured_by: aort_etrij2026
  page: 1
  decoding: not reported
results:
- id: r_avg_none_bipiatext
  evaluation_id: ev_avg_none_bipiatext
  quantity: R1
  coverage: direct
  metric_as_printed: average ASR, BIPIA-text
  value: '15.7'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 7
  location: Section 7.1
- id: r_avg_aort_bipiatext
  evaluation_id: ev_avg_aort_bipiatext
  quantity: R1
  coverage: direct
  metric_as_printed: average ASR, BIPIA-text
  value: '4.9'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 7
  location: Section 7.1
- id: r_avg_none_bipiacode
  evaluation_id: ev_avg_none_bipiacode
  quantity: R1
  coverage: direct
  metric_as_printed: average ASR, BIPIA-code
  value: '17.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 7
  location: Section 7.1
- id: r_avg_aort_bipiacode
  evaluation_id: ev_avg_aort_bipiacode
  quantity: R1
  coverage: direct
  metric_as_printed: average ASR, BIPIA-code
  value: '6.4'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 7
  location: Section 7.1
- id: r_avg_none_struqid
  evaluation_id: ev_avg_none_struqid
  quantity: R1
  coverage: direct
  metric_as_printed: average ASR, StruQ-ID
  value: '45.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 7
  location: Section 7.1
- id: r_avg_aort_struqid
  evaluation_id: ev_avg_aort_struqid
  quantity: R1
  coverage: direct
  metric_as_printed: average ASR, StruQ-ID
  value: '27.3'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 7
  location: Section 7.1
- id: r_avg_none_struqood
  evaluation_id: ev_avg_none_struqood
  quantity: R1
  coverage: direct
  metric_as_printed: average ASR, StruQ-OOD
  value: '45.0'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 7
  location: Section 7.1
- id: r_avg_aort_struqood
  evaluation_id: ev_avg_aort_struqood
  quantity: R1
  coverage: direct
  metric_as_printed: average ASR, StruQ-OOD
  value: '34.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 7
  location: Section 7.1
- id: r_l31_none_bipiatext
  evaluation_id: ev_l31_none_bipiatext
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, BIPIA-text
  value: '13.6'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 1
  location: abstract (Table 1 cell)
- id: r_l31_aort_bipiatext
  evaluation_id: ev_l31_aort_bipiatext
  quantity: R1
  coverage: direct
  metric_as_printed: ASR, BIPIA-text
  value: '2.1'
  unit: percent
  direction: lower_better
  qualifier: exact
  value_source: text
  page: 1
  location: abstract (Table 1 cell)
comparisons:
- id: c_avg_bipiatext
  left: ev_avg_aort_bipiatext
  right: ev_avg_none_bipiatext
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_avg_bipiacode
  left: ev_avg_aort_bipiacode
  right: ev_avg_none_bipiacode
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_avg_struqid
  left: ev_avg_aort_struqid
  right: ev_avg_none_struqid
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_avg_struqood
  left: ev_avg_aort_struqood
  right: ev_avg_none_struqood
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 7
- id: c_l31_bipiatext
  left: ev_l31_aort_bipiatext
  right: ev_l31_none_bipiatext
  purpose: W
  c1_base: matched
  c2_layout: matched
  c3_benchmark: matched
  c4_attacker: matched
  c5_judge: matched
  c6_decoding: matched
  varies: the defence (defended against undefended arm)
  page: 1
claims:
- id: k1
  text: AORT reduces ASR from 13.6% to 2.1% on LLaMA 3.1 8B on BIPIA-text and achieves the lowest ASR in most settings
  location: abstract
  anchor: AORT reduces the ASR from 13.6% to 2.1%
  jobs:
  - R1
  supported_by:
  - c_l31_bipiatext
  - c_avg_bipiatext
  - c_avg_bipiacode
  - c_avg_struqid
  - c_avg_struqood
  page: 1
coverage:
  R1: aggregate_only
  R2clean: not_reported
  R2attack: not_reported
  R3ns: not_reported
  R3cp: not_reported
  R3safe: not_reported
  where:
    R1: Section 7.1 averages; one per-model pair from the abstract
    R2clean: not reported in the main text
    R2attack: not reported
schema_gaps:
- where: results
  note: the PDF text layer encodes the decimal points of Table 1 as colons in every extraction (for example 13:6 for 13.6), so the per-model cells, including the same-data ISE and ASIDE baselines, cannot be checked against
    the text and are not coded; the text averages match the Table 1 cells read with points
- where: evaluations ev_*_none_*
  note: the Vanilla baseline is the same backbone fine-tuned on the same instruction data without the role-aware changes (same-data twin); the training data, success criteria and decoding are in the supplement, which was not
    read

## Stage 2. The PDFs read are the PDFs registered, and every report is an included study

In [ ]:
import csv, logging, datetime as dt
from collections import Counter
import schema, search, corpus, paths
for m in (schema, search, corpus, paths):
    importlib.reload(m)
logging.getLogger("pypdf").setLevel(logging.ERROR)
from pypdf import PdfReader

BATCH = "04g"
cfg_exclusions = 2
cfg_next = "batch 04h"
IDS = ['textpixels_arxiv2026', 'constsanitlog_icssa2026', 'crosslingjudge_ubmk2026', 'smallsecure_asiaccs2026', 'ragmitig_ccs2026', 'benchvalid_rs2026', 'localrag_applsci2026', 'ipidefrepl_computers2026', 'glasswall_techrxiv2026', 'aort_etrij2026']
man = {m["slug"]: m for m in search.read_csv(paths.REPORTS / "defences.csv")}
problems, texts = [], {}
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    pdf = paths.DATA_DEFENCES / f"{slug}.pdf"
    m = man.get(slug, {})
    if not pdf.exists():
        problems.append(f"{rid}: {pdf.name} not in data/defences"); continue
    sha = corpus.sha256(pdf)
    rd = PdfReader(str(pdf))
    texts[slug] = ([p.extract_text() or "" for p in rd.pages], [p.extract_text(extraction_mode="layout") or "" for p in rd.pages])
    ok_sha = sha == m.get("sha256")
    ok_inc = m.get("decision") == "include"
    print(f"{rid:<28} {slug:<12} pages {len(rd.pages):>3}  sha256 {'matches' if ok_sha else 'DIFFERS'}  decision {m.get('decision')}")
    if not ok_sha:
        problems.append(f"{rid}: PDF differs from the one registered in reports/defences.csv")
    if not ok_inc:
        problems.append(f"{rid}: not an included study")

## Stage 2b. Full-text exclusions found while coding

Each study below was included at title and abstract; its full text shows it does not meet the inclusion rule. The anchor is checked in its PDF, then the decision is written to schema/fulltext_decisions.csv and schema/screening_decisions.csv and carried into reports/screening.csv, reports/defences.csv, reports/fulltext_check.csv and reports/prisma_counts.json, as notebook 02d did.

In [ ]:
import json
import overlap
importlib.reload(overlap)
EXCL = [{'key': 'doi:10.20944/preprints202609.1881.v1', 'slug': 'doi_10_20944_preprints202609_1881_v1', 'reason': 'system level', 'anchor': 'Filter + Sandbox: Adding tool-call restrictions via allowlists', 'note': 'SafeAgent: every configuration tested contains a regex sanitiser with a trained classifier and, beyond Filter Only, a tool-call allowlist; the data markers and the safety-check prompt are never measured on their own'}, {'key': 'doi:10.5281/zenodo.18438220', 'slug': 'doi_10_5281_zenodo_18438220', 'reason': 'system level', 'anchor': 'after passing through a random 3-5 hop translation chain with defensive prompt augmentation', 'note': 'a separate translation model rewrites the input before the victim model sees it (coded as system level, as MirrorGuard was); most of the corpus is direct injection and jailbreaks'}]
FT_FIELDS = ["key", "slug", "decision", "reason", "basis", "anchor", "note"]
SD_FIELDS = ["key", "decision", "reason", "stage", "access_status", "note", "title"]
CHECK_FIELDS = ["key", "slug", "decision", "reason", "basis", "page", "status", "anchor", "sha256", "note"]
ft = search.read_csv(paths.SCHEMA / "fulltext_decisions.csv")
sd = search.read_csv(paths.SCHEMA / "screening_decisions.csv")
fchk = search.read_csv(paths.REPORTS / "fulltext_check.csv")
mlist = search.read_csv(paths.REPORTS / "defences.csv")
for e in EXCL:
    m = next((x for x in mlist if x["slug"] == e["slug"]), None)
    pdf = paths.DATA_DEFENCES / f"{e['slug']}.pdf"
    if m is None or not pdf.exists():
        problems.append(f"exclusion {e['slug']}: not in reports/defences.csv or PDF missing"); continue
    sha = corpus.sha256(pdf)
    if sha != m.get("sha256"):
        problems.append(f"exclusion {e['slug']}: PDF differs from the one registered")
    if e["reason"] not in search.EXCLUSION_REASONS:
        problems.append(f"exclusion {e['slug']}: reason not in the fixed list")
    pages = overlap.pdf_page_texts(pdf)
    a = overlap.norm_text(e["anchor"])
    hits = [i + 1 for i, t in enumerate(pages) if a in t]
    status = "found" if hits else "anchor not found"
    if not hits:
        problems.append(f"exclusion {e['slug']}: anchor not found in the PDF")
    row = {"key": e["key"], "slug": e["slug"], "decision": "exclude", "reason": e["reason"], "basis": "full text", "anchor": e["anchor"], "note": e["note"]}
    ft = [r for r in ft if r["key"] != e["key"]] + [row]
    fchk = [r for r in fchk if r["key"] != e["key"]] + [dict(row, page=str(hits[0]) if hits else "", status=status, sha256=sha)]
    n = 0
    for r in sd:
        if r["key"] == e["key"]:
            r.update(decision="exclude", reason=e["reason"], stage="full_text", note="full text: " + e["note"]); n += 1
    if n != 1:
        problems.append(f"exclusion {e['slug']}: {n} rows for {e['key']} in schema/screening_decisions.csv")
    m.update(decision="exclude", reason=e["reason"], stage="full_text", note="full text: " + e["note"])
    print(f"{e['slug']:<12} excluded at full text ({e['reason']}); anchor {status}{' on page ' + str(hits[0]) if hits else ''}")
def write_lf(path, rows, fields):              # the schema files were written with plain newlines; keep them that way
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore", lineterminator="\n")
        w.writeheader()
        w.writerows({k: r.get(k) or "" for k in fields} for r in rows)
write_lf(paths.SCHEMA / "fulltext_decisions.csv", ft, FT_FIELDS)
write_lf(paths.SCHEMA / "screening_decisions.csv", sd, SD_FIELDS)
search.write_csv(paths.REPORTS / "fulltext_check.csv", fchk, CHECK_FIELDS)
search.write_csv(paths.REPORTS / "defences.csv", mlist, list(mlist[0].keys()))
rows = search.read_csv(paths.REPORTS / "screening.csv")
rows, more = search.apply_decisions(rows, sd)
problems += more
search.write_csv(paths.REPORTS / "screening.csv", rows, search.SCREEN_FIELDS)
prisma = json.loads((paths.REPORTS / "prisma_counts.json").read_text())
prisma.update(search.prisma_counts(search.read_csv(paths.REPORTS / "search_log.csv"), rows))
prisma["fulltext_anchors_found"] = sum(1 for r in fchk if r["status"].startswith("found"))
(paths.REPORTS / "prisma_counts.json").write_text(json.dumps(prisma, indent=1))
print(f"included now: {prisma['included']}; full text assessed: {prisma['full_text_assessed']}; "
      f"full-text exclusions: {json.dumps(prisma['excluded_full_text_by_reason'])}")

## Stage 3. Validate against the frozen schema and the PDFs; grade the comparisons; extra checks

In [ ]:
import re
import pypdfium2 as pdfium
_docs = {}
def pdfium_page(slug, page):
    if slug not in _docs:
        _docs[slug] = pdfium.PdfDocument(str(paths.DATA_DEFENCES / f"{slug}.pdf"))
    return _docs[slug][page - 1].get_textpage().get_text_range()
S1 = schema.load_schema(paths.SCHEMA / "record_schema.yaml")
check_rows, summary, residual, third_rows = [], [], [], []
for rid in IDS:
    rec = schema.load_record(paths.RECORDS / f"{rid}.yaml")
    slug = rec["report"]["slug"]
    plain, lay = texts.get(slug, (None, None))
    probs, notes = schema.validate_record(rec, S1, plain, lay)
    kept, third = [], []                       # a value pypdf glues to its neighbour is looked for in a third extraction (pypdfium2)
    for p in probs:
        m = re.match(r"results:(\S+): value '(.+)' not found on page (\d+)$", p)
        if m and schema.value_on_page(m.group(2), pdfium_page(slug, int(m.group(3)))):
            third.append(f"{m.group(1)} ({m.group(2)!r}, page {m.group(3)})")
        else:
            kept.append(p)
    probs = kept
    for kind in ("defences", "evaluations", "results", "comparisons", "claims"):          # ids unique within each kind
        dup = [i for i, n in Counter(r["id"] for r in rec.get(kind) or []).items() if n > 1]
        probs += [f"{kind}: id {i} used more than once" for i in dup]
    fig_kept, third_chart = 0, []
    for r in rec.get("results") or []:                                                     # chart values kept as printed
        if r.get("value_source") == "figure_only" and r.get("value") not in (None, ""):
            fig_kept += 1
            pg = int(r["page"]) - 1
            if plain and not (schema.value_on_page(r["value"], plain[pg]) or schema.value_on_page(r["value"], lay[pg])):
                if schema.value_on_page(r["value"], pdfium_page(slug, int(r["page"]))):
                    third_chart.append(f"{r['id']} ({r['value']!r}, page {r['page']}, chart label)")
                else:
                    probs.append(f"results:{r['id']}: chart value {r['value']!r} not found on page {r['page']}")
    third_rows += [{"report": rid, "status": "value found in the third extraction", "detail": t} for t in third + third_chart]
    grades = Counter(schema.grade(c) for c in rec.get("comparisons") or [])
    n_val = sum(1 for r in rec.get("results") or [] if r.get("value_source") != "figure_only")
    n_anc = sum(1 for k in ("defences", "claims") for r in rec.get(k) or [] if r.get("anchor"))
    summary.append({"batch": BATCH, "report": rid, "slug": slug, "role": rec["report"]["role"],
                    "defences": len(rec.get("defences") or []), "evaluations": len(rec.get("evaluations") or []),
                    "results": len(rec.get("results") or []), "values_checked": n_val, "chart_values_checked": fig_kept,
                    "anchors_checked": n_anc, "comparisons": len(rec.get("comparisons") or []),
                    "grade_A": grades["A"], "grade_B": grades["B"], "grade_C": grades["C"], "grade_U": grades["U"],
                    "claims": len(rec.get("claims") or []),
                    **{f"cov_{q}": rec["coverage"][q] for q in ("R1", "R2clean", "R2attack", "R3ns", "R3cp", "R3safe")},
                    "values_third_extraction": len(third), "chart_values_third_extraction": len(third_chart), "residual_notes": len(notes), "problems": len(probs)})
    check_rows += [{"report": rid, "status": "problem", "detail": p} for p in probs]
    residual += [{"report": rid, "where": n.split(": ", 1)[0], "note": n.split(": ", 1)[1]} for n in notes]
    print(f"{rid:<28} problems {len(probs)}; values {n_val} ({len(third)} via the third extraction); chart values {fig_kept} ({len(third_chart)} via the third extraction); anchors {n_anc}; grades {dict(grades)}")
    for p in probs:
        print("   ", p)
    problems += [f"{rid}: {p}" for p in probs]
check_rows += third_rows + [{"report": g["report"], "status": "residual note", "detail": f"{g['where']}: {g['note']}"} for g in residual]
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_check.csv", check_rows or [{"report": "all", "status": "ok", "detail": "no problems"}],
                 ["report", "status", "detail"])
search.write_csv(paths.REPORTS / f"extraction_{BATCH}_summary.csv", summary, list(summary[0].keys()))
for g in residual:
    print(f"  residual note, {g['report']}: {g['note'][:150]}")

## Stage 4. Batch register (one row per coded report, kept across batches)

In [ ]:
reg_path = paths.REPORTS / "extraction_batches.csv"
reg = [r for r in search.read_csv(reg_path) if r["batch"] != BATCH] if reg_path.exists() else []
today = dt.date.today().isoformat()
for s in summary:
    reg.append({"batch": BATCH, "report": s["report"], "slug": s["slug"], "coded": today,
                "status": "checked" if s["problems"] == 0 else "problems"})
reg.sort(key=lambda r: (r["batch"], r["report"]))
search.write_csv(reg_path, reg, ["batch", "report", "slug", "coded", "status"])
included = sum(1 for m in search.read_csv(paths.REPORTS / "defences.csv") if m.get("decision") == "include")   # read again: Stage 2b may change it
pilot = len(list(paths.RECORDS.glob("*.yaml"))) - len({r["report"] for r in reg})
print(f"records in reports/records: {len(list(paths.RECORDS.glob('*.yaml')))} (pilot {pilot}, batches {len({r['report'] for r in reg})}); included studies: {included}")

## Stage 5. Decision record and tests

In [ ]:
tot = {k: sum(s[k] for s in summary) for k in ("defences", "evaluations", "results", "values_checked", "values_third_extraction", "chart_values_checked",
                                              "chart_values_third_extraction", "anchors_checked", "comparisons", "grade_A", "grade_B", "grade_C", "grade_U", "claims")}
status = "BATCH CHECKED" if not problems else "BATCH NOT CHECKED: PROBLEMS BELOW"
lines = [f"decision_{BATCH}: {status}", f"this step: {dt.date.today().isoformat()}",
         "reports: " + ", ".join(IDS),
         f"records: {tot['defences']} defences, {tot['evaluations']} evaluations, {tot['results']} results, "
         f"{tot['comparisons']} comparisons, {tot['claims']} claims",
         f"checked against the PDFs: {tot['values_checked']} printed values ({tot['values_third_extraction']} of them found only in the third "
         f"extraction), {tot['chart_values_checked']} chart values ({tot['chart_values_third_extraction']} of them found only in the third extraction) "
         f"and {tot['anchors_checked']} anchors, each found on its page",
         f"comparison grades: A {tot['grade_A']}, B {tot['grade_B']}, C {tot['grade_C']}, U {tot['grade_U']}",
         f"residual notes: {len(residual)}; full-text exclusions found while coding: {cfg_exclusions}",
         f"next: {cfg_next}"] + [f"- problem: {p}" for p in problems]
(paths.REPORTS / f"decision_{BATCH}.txt").write_text("\n".join(lines) + "\n")
print("\n".join(lines))
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "tests"], capture_output=True, text=True)
print(tests.stdout.strip().splitlines()[-1] if tests.stdout.strip() else tests.stderr.strip()[-400:])

## Stage 6. Commit

In [ ]:
import os, subprocess
os.chdir(os.environ.get("SOURCEMAP_REPO_ROOT", "/content/drive/MyDrive/SourceMap_Research/sourcemap-research"))
r = subprocess.run(["python", "tools/commit_cell.py",
                    "04g extraction batch g: ten included studies coded from their PDFs and two full-text exclusions, values and anchors checked, comparisons graded"],
                   capture_output=True, text=True)
print(r.stdout)
print(r.stderr)